In [ ]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import data_utils
from base_utils_qwen import competition_scorer as bfrb_competition_scorer, evaluate_holdout, plot_training_curves
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import train_test_split

from baselines_utils import (
    make_baseline_pipeline,
    build_feature_extractor,
    build_classifier,
    RidgeRocketClassifier,
    HAS_TF,
)

In [ ]:
# ============================================================
# CONFIGURATION — switch feature + classifier pipelines here
# ============================================================
TRAIN_DUMMY = False
TRAIN_RF = True
TRAIN_ROCKET = False
TRAIN_CNN = False  # requires tensorflow

# Feature extraction mode per model family:
#   tabular_simple | tabular_honeycomb | temporal_honeycomb | temporal_raw
FEATURE_MODE_TABULAR = "tabular_honeycomb"
FEATURE_MODE_ROCKET = "temporal_honeycomb"
FEATURE_MODE_CNN = "temporal_honeycomb"

# Tabular augmentation (RF only — applied inside pipeline when True)
use_tabular_augment = False
augment_kwargs = {"use_gaussian_noise": True, "noise_std": 0.01}
baseline_kwargs = {}  # Empty - no longer used for feature kwargs


target_col = "gesture_action"
search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
train_size = 0.7  # lower for quick local tests; raise for full runs
error_score_constant = 0.0
verbose = 4
do_cross_val = False

# Use eg.csv sample for fast smoke tests (set False for full train.csv)
use_eg_sample = False
eg_sample_pct = 0.02

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if target_col == 'bfrb':
    competition_scorer = bfrb_competition_scorer
else:
    competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

filter_non_brb_classes = True  
filter_orientation_class_list = ['Seated Straight'] # 'Seated Lean Non Dom - FACE DOWN', 'Lie on Side - Non Dominant', 'Lie on Back', 'Seated Straight'

In [ ]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    print(f"Using eg.csv sample: {raw_train_df['sequence_id'].nunique()} sequences")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(frac=eg_sample_pct, random_state=random_state)
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()
        print(f"Using {eg_sample_pct:.0%} sequence sample: {raw_train_df['sequence_id'].nunique()} sequences")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
train_df = train_df.drop(columns=["handedness"])

# Targets
train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

if filter_non_brb_classes:
    train_df = train_df.loc[train_df['is_target'],:]

if filter_orientation_class_list is not None:
    train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]

    # Get unique sequences
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()

    # Stratified split by your target column
    train_seqs, test_seqs = train_test_split(
        sequences['sequence_id'], 
        test_size=(1 - train_size), 
        stratify=sequences[target_col],  # or use multiple columns
        random_state=random_state
    )

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]
else:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
    )

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

In [ ]:
# ============================================================================
# PARAMETER SPACE DEFINITION — INCLUDES FEATURE EXTRACTION
# ============================================================================

if search_mode == "bayesian":
    
    # ===== Random Forest (Tabular Honeycomb) =====
    rf_param_space = {
        # ---- Feature extraction (Honeycomb) ----
        "extractor__acc_modes": Categorical([
            "raw", "raw|velocity", "smoothed|velocity|jerk"
        ]),
        "extractor__rotation_modes": Categorical([
            "quaternion", "quaternion|angular_velocity", "quaternion|euler"
        ]),
        "extractor__tof_modes": Categorical([
            "sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"
        ]),
        "extractor__thm_modes": Categorical([
            "centered_diff", "diff", "centered"
        ]),
        "extractor__sampling_rate": Integer(10, 200),
        "extractor__window_size": Integer(5, 50),
        "extractor__clip_value": Categorical([None, 50.0, 100.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        # ---- Random Forest classifier ----
        "classifier__base_estimator__n_estimators": Integer(10, 500),
        "classifier__base_estimator__max_depth": Integer(5, 25),
        "classifier__base_estimator__min_samples_leaf": Integer(1, 10),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }
    
    # ===== MiniRocket (Temporal Honeycomb) =====
    rocket_param_space = {
        # ---- Feature extraction (SequenceTensorExtractor) ----
        "extractor__acc_modes": Categorical([
            "raw", "raw|velocity", "smoothed|velocity|jerk"
        ]),
        "extractor__rotation_modes": Categorical([
            "quaternion", "quaternion|angular_velocity", "quaternion|euler"
        ]),
        "extractor__tof_modes": Categorical([
            "sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"
        ]),
        "extractor__thm_modes": Categorical([
            "centered_diff", "diff", "centered"
        ]),
        "extractor__sampling_rate": Integer(10, 200),
        "extractor__maxlen": Integer(30, 200),
        "extractor__window_size": Integer(5, 50),
        "extractor__clip_value": Categorical([None, 50.0, 100.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
        # ---- MiniRocket classifier ----
        "classifier__base_estimator__num_kernels": Integer(84, 5000),
        "classifier__base_estimator__alpha": Real(1e-4, 1e2, prior="log-uniform"),
        "classifier__base_estimator__feature_selection_percentile": Categorical([None, 25, 50]),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }
    
    # ===== 1D CNN (Temporal Honeycomb) =====
    cnn_param_space = {
        # ---- Feature extraction (SequenceTensorExtractor) ----
        "extractor__acc_modes": Categorical([
            "raw", "raw|velocity", "smoothed|velocity|jerk"
        ]),
        "extractor__rotation_modes": Categorical([
            "quaternion", "quaternion|angular_velocity", "quaternion|euler"
        ]),
        "extractor__tof_modes": Categorical([
            "sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"
        ]),
        "extractor__thm_modes": Categorical([
            "centered_diff", "diff", "centered"
        ]),
        "extractor__sampling_rate": Integer(10, 200),
        "extractor__maxlen": Integer(60, 200),
        "extractor__window_size": Integer(5, 50 ),
        "extractor__clip_value": Categorical([None, 50.0, 100.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
        # ---- CNN classifier ----
        "classifier__base_estimator__filters": Categorical(["32-64", "64-128", "128-256"]),
        "classifier__base_estimator__kernels": Categorical(["3-3", "5-3", "7-5-3"]),
        "classifier__base_estimator__pools": Categorical(["none", "2", "2-2"]),
        "classifier__base_estimator__dropout": Real(0.1, 0.5),
        "classifier__base_estimator__spatial_dropout": Real(0.0, 0.3),
        "classifier__base_estimator__l2_reg": Real(1e-5, 1e-2, prior="log-uniform"),
        "classifier__base_estimator__learning_rate": Real(1e-4, 1e-2, prior="log-uniform"),
        "classifier__base_estimator__batch_size": Categorical([16, 32, 64]),
        "classifier__base_estimator__epochs": Categorical([100]),
        "classifier__base_estimator__patience": Categorical([10]),
    }

else:  # GRID SEARCH
    
    # ===== Random Forest (Tabular Honeycomb) =====
    rf_param_space = {
        "extractor__acc_modes": ["smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__sampling_rate": [20],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "classifier__base_estimator__n_estimators": [300],
        "classifier__base_estimator__max_depth": [30],
        "classifier__base_estimator__class_weight": ["balanced"],
    }
    
    # ===== MiniRocket (Temporal Honeycomb) =====
    rocket_param_space = {
        "extractor__acc_modes": ["raw", "smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__sampling_rate": [20, 100],
        "extractor__maxlen": [120, 200],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "classifier__base_estimator__num_kernels": [1000, 3000],
        "classifier__base_estimator__alpha": [5.0],
        "classifier__base_estimator__feature_selection_percentile": [20],
        "classifier__base_estimator__class_weight": ["balanced"],
    }
    
    # ===== 1D CNN (Temporal Honeycomb) =====
    cnn_param_space = {
        "extractor__acc_modes": ["raw", "smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__sampling_rate": [20, 100],
        "extractor__maxlen": [120],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "classifier__base_estimator__filters": ["32-64"],
        "classifier__base_estimator__kernels": ["3-3"],
        "classifier__base_estimator__pools": ["2"],
        "classifier__base_estimator__dropout": [0.3],
        "classifier__base_estimator__spatial_dropout": [0.1],
        "classifier__base_estimator__l2_reg": [1e-4],
        "classifier__base_estimator__learning_rate": [5e-3],
        "classifier__base_estimator__batch_size": [32],
        "classifier__base_estimator__epochs": [20],
    }

In [ ]:
results_list = []
fitted_models = {}

# ============================================================
# BASELINE 1: DUMMY CLASSIFIER (tabular)
# ============================================================
if TRAIN_DUMMY:
    print("\n--- Training Baseline 1: Dummy Classifier ---")
    dummy_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_TABULAR,
        classifier_name="dummy",
        target_col=target_col,
        feature_kwargs={},  # Empty — will be searched
        random_state=random_state,
    )
    dummy_pipe.fit(X_train, y_train)
    fitted_models["dummy"] = dummy_pipe
    dummy_eval = evaluate_holdout(y_test, dummy_pipe.predict(X_test), target_col=target_col)
    results_list.append({"Model": "Dummy", "CV Score": np.nan, "Holdout Score": dummy_eval["competition_score"]})

# ============================================================
# BASELINE 2: RANDOM FOREST (tabular + optional augment)
# ============================================================
if TRAIN_RF:
    print("\n--- Training Baseline 2: Random Forest ---")
    rf_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_TABULAR,
        classifier_name="rf",
        target_col=target_col,
        augment=use_tabular_augment,
        augment_kwargs=augment_kwargs,
        feature_kwargs={},  # Empty — will be searched
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rf_search = BayesSearchCV(
            rf_pipe, rf_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose,
        )
    else:
        rf_search = GridSearchCV(
            rf_pipe, rf_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose,
        )

    rf_search.fit(X_train, y_train, groups=groups)
    fitted_models["rf"] = rf_search.best_estimator_
    rf_eval = evaluate_holdout(y_test, rf_search.predict(X_test), target_col=target_col, verbose=True)
    results_list.append({
        "Model": "Random Forest",
        "CV Score": rf_search.best_score_,
        "Holdout Score": rf_eval["competition_score"],
        "Best Params": rf_search.best_params_,
    })
    print(f"RF Best CV Score: {rf_search.best_score_:.4f} | Holdout: {rf_eval['competition_score']:.4f}")

# ============================================================
# BASELINE 3: MINIROCKET (temporal tensors + Ridge regularisation)
# ============================================================
if TRAIN_ROCKET:
    print("\n--- Training Baseline 3: MiniRocket (Ridge) ---")
    rocket_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_ROCKET,
        classifier_name="ridge_rocket",
        target_col=target_col,
        feature_kwargs={},  # Empty — will be searched
        classifier_kwargs={"num_kernels": 500, "alpha": 1.0},
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rocket_search = BayesSearchCV(
            rocket_pipe, rocket_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose,
        )
    else:
        rocket_search = GridSearchCV(
            rocket_pipe, rocket_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose,
        )

    rocket_search.fit(X_train, y_train, groups=groups)
    fitted_models["rocket"] = rocket_search.best_estimator_
    rocket_eval = evaluate_holdout(y_test, rocket_search.predict(X_test), target_col=target_col, verbose=True)
    results_list.append({
        "Model": "MiniRocket",
        "CV Score": rocket_search.best_score_,
        "Holdout Score": rocket_eval["competition_score"],
        "Best Params": rocket_search.best_params_,
    })
    print(f"MiniRocket CV: {rocket_search.best_score_:.4f} | Holdout: {rocket_eval['competition_score']:.4f}")

# ============================================================
# BASELINE 4: TEMPORAL 1D CNN
# ============================================================
if TRAIN_CNN and HAS_TF:
    print("\n--- Training Baseline 4: Temporal 1D CNN ---")
    cnn_pipe = make_baseline_pipeline(
        feature_mode=FEATURE_MODE_CNN,
        classifier_name="cnn",
        target_col=target_col,
        feature_kwargs={},  # Empty — will be searched
        classifier_kwargs={"epochs": 20, "verbose": 1, "filters": "32-64"},
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        cnn_search = BayesSearchCV(
            cnn_pipe, cnn_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose,
        )
    else:
        cnn_search = GridSearchCV(
            cnn_pipe, cnn_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose,
        )

    cnn_search.fit(X_train, y_train, groups=groups)
    fitted_models["cnn"] = cnn_search.best_estimator_
    cnn_eval = evaluate_holdout(y_test, cnn_search.predict(X_test), target_col=target_col, verbose=False)
    results_list.append({
        "Model": "Temporal CNN",
        "CV Score": cnn_search.best_score_,
        "Holdout Score": cnn_eval["competition_score"],
        "Best Params": cnn_search.best_params_,
    })
    print(f"CNN CV: {cnn_search.best_score_:.4f} | Holdout: {cnn_eval['competition_score']:.4f}")
elif TRAIN_CNN and not HAS_TF:
    print("Skipping CNN — tensorflow not installed")

In [ ]:
# ============================================================
# FINAL SUMMARY + BEST MODEL HOLDOUT EVAL
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "=" * 50)
print("BASELINES SUMMARY")
print("=" * 50)
print(results_df.to_string(index=False))

# Full report for best holdout model
if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"].lower().replace(" ", "_")
    key_map = {"dummy": "dummy", "random_forest": "rf", "minirocket": "rocket", "temporal_cnn": "cnn"}
    model_key = key_map.get(best_name, list(fitted_models.keys())[0])
    if model_key in fitted_models:
        print(f"\nDetailed holdout eval for best model: {results_df.iloc[0]['Model']}")
        evaluate_holdout(y_test, fitted_models[model_key].predict(X_test), target_col=target_col)

In [53]:
import pandas as pd
import os
import sys

import matplotlib.pyplot as plt

current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)
src_path = os.path.join(workspace_root, "src")
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

try:
    import data_utils
    from base_utils_qwen import prepare_bayesian_space, competition_scorer as bfrb_competition_scorer, evaluate_holdout, SequenceExtractor
    from proto_utils_v4 import V4PrototypicalNetwork, V4MultiHeadPrototypicalNetwork
    print("✅ Imports loaded successfully")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise

✅ Imports loaded successfully


In [ ]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)


✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data


In [12]:
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

In [40]:
child_rows = train_demo_df['age']  <= 18

train_demo_df.loc[child_rows, 'age_band'] = 'child'
train_demo_df.loc[(train_demo_df['age'] > 18) & (train_demo_df['age'] < 25), 'age_band'] = 'young_adult'
train_demo_df.loc[train_demo_df['age'] >= 25, 'age_band'] = 'adult'

train_demo_df.groupby(['handedness', 'age_band']).agg(**{
    'subjects':('subject','unique'),
    'subject count':('subject', 'nunique')})

subjects  \
handedness age_band                                                         
0          adult        [SUBJ_002923, SUBJ_013623, SUBJ_019756, SUBJ_0...   
           child        [SUBJ_028998, SUBJ_039234, SUBJ_055211, SUBJ_0...   
           young_adult                         [SUBJ_032233, SUBJ_041243]   
1          adult        [SUBJ_000206, SUBJ_003328, SUBJ_011323, SUBJ_0...   
           child        [SUBJ_001430, SUBJ_004117, SUBJ_008304, SUBJ_0...   
           young_adult  [SUBJ_012088, SUBJ_019297, SUBJ_020948, SUBJ_0...   

                        subject count  
handedness age_band                    
0          adult                    4  
           child                    4  
           young_adult              2  
1          adult                   26  
           child                   35  
           young_adult             10

In [14]:
train_df.groupby(['bfrb']).agg(**{
    'Sequence Count':('sequence_id','nunique')
})

,Sequence Count
bfrb,
Above ear - pull hair,638
Cheek - pinch skin,637
Eyebrow - pull hair,638
Eyelash - pull hair,640
Forehead - pull hairline,640
Forehead - scratch,640
Neck - pinch skin,640
Neck - scratch,640
non_bfrb,3038


In [78]:
adult_right_subject = 'SUBJ_012088'
adult_right_sequence_id = 'SEQ_000063'

adult_right_sequence_df = train_df[train_df['subject'] == adult_right_subject]
adult_right_sequence_df = adult_right_sequence_df[adult_right_sequence_df['sequence_id'] == adult_right_sequence_id]
adult_right_sequence_df.head()

,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,acc_y,...,tof_5_v58,tof_5_v59,tof_5_v60,tof_5_v61,tof_5_v62,tof_5_v63,gesture_position,gesture_action,is_target,bfrb
row_id,,,,,,,,,,,,,,,,,,,,,
SEQ_000063_000000,Target,SEQ_000063,0,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.363281,-4.378906,...,230.0,238.0,230.0,219.0,214.0,212.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000001,Target,SEQ_000063,1,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.402344,-4.378906,...,228.0,238.0,236.0,224.0,214.0,216.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000002,Target,SEQ_000063,2,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.324219,-4.417969,...,238.0,235.0,234.0,221.0,215.0,214.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000003,Target,SEQ_000063,3,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.359375,-4.378906,...,230.0,237.0,233.0,224.0,215.0,213.0,Eyelash,pull hair,True,Eyelash - pull hair
SEQ_000063_000004,Target,SEQ_000063,4,SUBJ_012088,Seated Lean Non Dom - FACE DOWN,Moves hand to target location,Transition,Eyelash - pull hair,7.398438,-4.378906,...,226.0,239.0,235.0,222.0,218.0,213.0,Eyelash,pull hair,True,Eyelash - pull hair


In [98]:
sequence_extractor = SequenceExtractor(
    acc_modes = 'raw'
)

transformed = sequence_extractor.fit_transform(adult_right_sequence_df)

In [ ]:
transformed.imu

,acc_modes,'raw'
,use_acc_magnitude,False
,use_linear_acc_magnitude,False
,window_size,7
,smooth_alpha,None
,sequence_col,'sequence_id'


In [82]:
transformed.keys()

dict_keys(['X', 'sequence_ids'])

In [68]:
transformed

{'X': array([[[ 7.3632812e+00, -4.3789062e+00,  5.2500000e+00, ...,
           2.9733628e-01,  7.9405330e-02,  8.9463927e-02],
         [ 7.4023438e+00, -4.3789062e+00,  5.3632812e+00, ...,
           2.9733628e-01,  7.9405330e-02,  8.9463927e-02],
         [ 7.3242188e+00, -4.4179688e+00,  5.3242188e+00, ...,
           2.9733628e-01,  7.9405330e-02,  8.9463927e-02],
         ...,
         [-9.9900000e+02, -9.9900000e+02, -9.9900000e+02, ...,
          -9.9900000e+02, -9.9900000e+02, -9.9900000e+02],
         [-9.9900000e+02, -9.9900000e+02, -9.9900000e+02, ...,
          -9.9900000e+02, -9.9900000e+02, -9.9900000e+02],
         [-9.9900000e+02, -9.9900000e+02, -9.9900000e+02, ...,
          -9.9900000e+02, -9.9900000e+02, -9.9900000e+02]]],
       shape=(1, 128, 138), dtype=float32),
 'sequence_ids': array(['SEQ_000063'], dtype='<U10')}

In [20]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import classification_report

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
try:
    import sktime
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "sktime", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import data_utils
from baselines_utils import (
    evaluate_holdout
)
from base_utils_qwen import (
    SequenceExtractor,
    competition_scorer as bfrb_competition_scorer,
    evaluate_holdout
)
from ensemble_utils import HierarchicalBFRBEnsemble
from single_minirocket import SingleMiniRocketClassifier
from sklearn.metrics import f1_score, make_scorer

In [21]:
# ============================================================
# CONFIGURATION
# ============================================================
TRAIN_SINGLE = True   # Set to True to train/evaluate the Single MiniRocket
TRAIN_ENSEMBLE = False # Set to True to train/evaluate the Hierarchical Ensemble

preprocess_handness = False

# Options: "bfrb", "orientation", "gesture", "gesture_action", "gesture_position", "is_target", "phase"
# Note: Ensure the chosen column exists in your train_df before running!
TARGET_COL = "bfrb"  
orientation_col = "orientation"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 3
n_iter = 10
train_size = 0.2
error_score_constant = 0.0
verbose = 3
do_cross_val = False

full_wham = True
if full_wham:
    test_size = 1 - train_size
else:
    test_size = min(0.25, 1 - train_size)

cv_object = GroupKFold(n_splits=n_splits) if do_cross_val else GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# DYNAMIC SCORER SELECTION
if TARGET_COL == 'bfrb':
    scorer = bfrb_competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

print(f"TRAIN_SINGLE: {TRAIN_SINGLE} | TRAIN_ENSEMBLE: {TRAIN_ENSEMBLE}")
print(f"Target Column: {TARGET_COL}")
print(f"Search mode: {search_mode}")

TRAIN_SINGLE: True | TRAIN_ENSEMBLE: False
Target Column: bfrb
Search mode: grid


In [22]:
data_root = data_utils.find_data_root()
raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if preprocess_handness: 
# Handedness & Upside-down corrections
    if "handedness" in train_demo_df.columns:
        train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
        left_handed_mask = train_df["handedness"].eq(0)
        train_df.loc[left_handed_mask, "acc_x"] *= -1.0
        train_df = train_df.drop(columns=["handedness"])

    upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
    train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
    train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Create alternative target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]


Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [23]:
# Get unique sequences for splitting to prevent leakage
sequences = train_df[['sequence_id', 'is_target', TARGET_COL, orientation_col]].drop_duplicates()
seq_ids = sequences['sequence_id'].unique()

if not do_cross_val:
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(seq_ids, groups=seq_ids))
    train_seqs = seq_ids[train_idx]
    test_seqs = seq_ids[test_idx]
else:
    train_seqs = seq_ids # For CV, we use all data in the search object
    test_seqs = []

train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy() if len(test_seqs) > 0 else pd.DataFrame()

X_train = train_sample_df
X_test = hold_out_df

# Setup y based on MODE
if TRAIN_ENSEMBLE:
    y_train = train_sample_df[["sequence_id", "is_target", orientation_col, TARGET_COL]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", orientation_col, TARGET_COL]].copy() if len(hold_out_df) > 0 else pd.DataFrame()
else:
    y_train = train_sample_df[["sequence_id", "is_target", TARGET_COL]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", TARGET_COL]].copy() if len(hold_out_df) > 0 else pd.DataFrame()

groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique() if len(X_test) > 0 else 0}")

Train sequences: 1630 | Test sequences: 6521


In [24]:
# ============================================================
# PARAMETER SPACE DEFINITION
# ============================================================
if TRAIN_ENSEMBLE:
    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        bayes_extractor_space = {
            "extractor__acc_modes": Categorical(["smoothed|velocity|jerk"]),
            "extractor__rotation_modes": Categorical([ "quaternion|angular_velocity|euler"]),
            "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats"]),
            "extractor__thm_modes": Categorical(["centered_diff"]),
            "extractor__sampling_rate": Categorical([200]),
            "extractor__maxlen": Categorical([180]),
            "extractor__window_size": Categorical([50]),
            "extractor__clip_value": Categorical([150.0]),
            "extractor__interp_mode": Categorical(["linear"]),
            "extractor__motion_filter_mode": Categorical([ "kalman"]),
            "extractor__use_dead_reckoning": Categorical([True]),
            "extractor__dead_reckoning_detrend": Categorical([True]),
            "extractor__kalman_process_noise": Real(1e-6, 1e-1, prior="log-uniform"),
            "extractor__kalman_measurement_noise": Real(1e-3, 1e2, prior="log-uniform"),
        }

        ensemble_param_space = {
            **bayes_extractor_space,
            # Layer 1: Binary BFRB
            "classifier__l1_num_kernels": Categorical([3700]),
            # "classifier__l1_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l1_class_weight": Categorical(["balanced", None]),
            # "classifier__l1_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            # Layer 2: Orientation
            "classifier__l2_num_kernels": Categorical([3700]),
            # "classifier__l2_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l2_class_weight": Categorical(["balanced", None]),
            # "classifier__l2_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            # Layer 3: BFRB per Orientation (Models 1-4)
            "classifier__l3_1_num_kernels": Integer(1000, 3000),
            # "classifier__l3_1_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_1_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            "classifier__l3_2_num_kernels": Integer(1000, 3000),
            # "classifier__l3_2_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_2_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            "classifier__l3_3_num_kernels": Integer(1000, 3000),
            # "classifier__l3_3_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_3_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            "classifier__l3_4_num_kernels": Integer(1000, 3000),
            # "classifier__l3_4_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_4_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            # "classifier__l3_class_weight": Categorical(["balanced", None]),
        }

    else:  # GRID SEARCH
        grid_extractor_space = {
            "extractor__acc_modes": ["smoothed|velocity|displacement|jerk"],
            "extractor__rotation_modes": ["quaternion|angular_velocity|euler"], # FIXED from "raw"
            "extractor__tof_modes": ["pooled_stats|sensor_stats"],                     # FIXED from "raw"
            "extractor__thm_modes": ["centered_diff|diff"],                              # FIXED typo from "raww"
            "extractor__sampling_rate": [200],
            "extractor__maxlen": [160],
            "extractor__window_size": [40],
            "extractor__clip_value": [None],
            "extractor__interp_mode": ["linear"],
            "extractor__motion_filter_mode": [None],
            "extractor__use_dead_reckoning": [False],
            "extractor__dead_reckoning_detrend": [False],
            "extractor__kalman_process_noise": [1e-3],
            "extractor__kalman_measurement_noise": [1e-1],
        }

        ensemble_param_space = {
            **grid_extractor_space,
            # Layer 1
            "classifier__l1_num_kernels": [2000],
            "classifier__l1_alpha": [1e3],
            "classifier__l1_class_weight": ["balanced"],
            "classifier__l1_feature_selection_percentile": [50], # ADDED
            
            # Layer 2
            "classifier__l2_num_kernels": [2000],
            "classifier__l2_alpha": [1e4],
            "classifier__l2_class_weight": ["balanced"],
            "classifier__l2_feature_selection_percentile": [50], # ADDED
            
            # Layer 3
            "classifier__l3_1_num_kernels": [2000],
            "classifier__l3_1_alpha": [1e4],
            "classifier__l3_1_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_2_num_kernels": [2000],
            "classifier__l3_2_alpha": [1e4],
            "classifier__l3_2_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_3_num_kernels": [2000],
            "classifier__l3_3_alpha": [1e4],
            "classifier__l3_3_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_4_num_kernels": [2000],
            "classifier__l3_4_alpha": [1e4],
            "classifier__l3_4_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_class_weight": ["balanced"],
        }

# ... (Keep your existing bayes_extractor_space / grid_extractor_space definitions) ...

if TRAIN_SINGLE:
# ============================================================
# SINGLE MINI-ROCKET PARAMETER SPACE
# ============================================================
    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        # Base Extractor Space (Bayesian)
        bayes_extractor_space = {
            "extractor__acc_modes": Categorical(["raw", "smoothed|velocity|jerk"]),
            "extractor__rotation_modes": Categorical(["quaternion|angular_velocity|euler", "raw"]),
            "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats"]),
            "extractor__thm_modes": Categorical(["centered_diff"]),
            "extractor__sampling_rate": Integer(20, 200),
            "extractor__maxlen": Integer(100, 200),
            "extractor__window_size": Integer(10, 50),
            "extractor__clip_value": Real(30.0, 100.0, prior="linear"),
            "extractor__interp_mode": Categorical(["linear"]),
            "extractor__motion_filter_mode": Categorical(["kalman", None]),
            "extractor__use_dead_reckoning": Categorical([False]),
            "extractor__dead_reckoning_detrend": Categorical([False]),
            "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
            "extractor__kalman_measurement_noise": Real(1e-2, 1e-1, prior="log-uniform"),
            "extractor__padding_value": Categorical([0.0]), # CRITICAL for MiniRocket
        }

        # Merge with Classifier Space
        single_param_space = {
            **bayes_extractor_space,
            "classifier__target_col": Categorical([TARGET_COL]),
            "classifier__num_kernels": Integer(1000, 2000),
            "classifier__alpha": Real(1e1, 1e4, prior="log-uniform"),
            "classifier__feature_selection_percentile": Categorical([None, 25, 50, 75]),
            "classifier__class_weight": Categorical(["balanced", None]),
        }

    else:  # GRID SEARCH
        # Base Extractor Space (Grid)
        grid_extractor_space = {
            # "extractor__acc_modes": ["smoothed|velocity|displacement|jerk"],
            # "extractor__rotation_modes": ["quaternion|angular_velocity|euler"],
            # "extractor__tof_modes": ["pooled_stats|sensor_stats"],
            # "extractor__thm_modes": ["centered_diff|diff"],
            "extractor__acc_modes": ["raw"],
            "extractor__rotation_modes": ["raw"],
            "extractor__tof_modes": ["raw"],
            "extractor__thm_modes": ["raw"],
            "extractor__sampling_rate": [20],
            "extractor__maxlen": [180],
            "extractor__window_size": [20],
            "extractor__clip_value": [None],
            "extractor__interp_mode": ["linear"],
            "extractor__motion_filter_mode": [None],
            "extractor__use_dead_reckoning": [False],
            "extractor__dead_reckoning_detrend": [False],
            "extractor__kalman_process_noise": [1e-3],
            "extractor__kalman_measurement_noise": [1e-1],
            "extractor__padding_value": [0.0], # CRITICAL for MiniRocket
        }

        # Merge with Classifier Space
        single_param_space = {
            **grid_extractor_space,
            "classifier__target_col": [TARGET_COL],
            "classifier__num_kernels": [84*10],
            "classifier__alpha": [1e3],
            "classifier__feature_selection_percentile": [None],
            "classifier__class_weight": ["balanced"],
        }

In [25]:
# ============================================================
# MODEL TRAINING & EVALUATION LOOP
# ============================================================
results_list = []
fitted_models = {}

# -----------------------------------------------------------------
# 1. SINGLE MINI-ROCKET MODEL
# -----------------------------------------------------------------
if TRAIN_SINGLE:
    print(f"\n--- Training: Single MiniRocket (Target: {TARGET_COL}) ---")
    
    single_pipe = Pipeline([
        ("extractor", SequenceExtractor(
            acc_modes="smoothed|velocity|jerk",
        )),
        ("classifier", SingleMiniRocketClassifier(
            target_col=TARGET_COL,
            num_kernels=2000,
            alpha=1e3,
            feature_selection_percentile=50,
            class_weight="balanced",
            random_state=random_state
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        single_search = BayesSearchCV(
            single_pipe, single_param_space, n_iter=n_iter, scoring=scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        single_search = GridSearchCV(
            single_pipe, single_param_space, scoring=scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    single_search.fit(X_train, y_train, groups=groups)
    fitted_models["Single MiniRocket"] = single_search.best_estimator_
    
    y_pred_single = single_search.predict(X_test)
    
    # Dynamic Evaluation based on TARGET_COL
    if TARGET_COL == 'bfrb':
        eval_dict_single = evaluate_holdout(y_test, y_pred_single, target_col=TARGET_COL, verbose=True)
        score_single = eval_dict_single.get("competition_score", 0)
    else:
        from sklearn.metrics import f1_score, classification_report
        y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
        print("\nClassification Report:")
        print(classification_report(y_test_seq[TARGET_COL], y_pred_single, zero_division=0))
        score_single = f1_score(y_test_seq[TARGET_COL], y_pred_single, average="macro", zero_division=0)
        print(f"Macro F1 Score: {score_single:.4f}")
    
    print(f"Single MiniRocket Best CV Score: {single_search.best_score_:.4f} | Holdout Score: {score_single:.4f}")
    print(f"Best Params: {single_search.best_params_}")
    
    results_list.append({
        "Model": "Single MiniRocket",
        "Target": TARGET_COL,
        "CV Score": single_search.best_score_,
        "Holdout Score": score_single,
        "Best Params": single_search.best_params_,
    })

# -----------------------------------------------------------------
# 2. HIERARCHICAL ENSEMBLE MODEL (Strictly for BFRB)
# -----------------------------------------------------------------
if TRAIN_ENSEMBLE:
    print("\n--- Training: Hierarchical BFRB Ensemble ---")
    
    l3_override_dict = None 

    ensemble_pipe = Pipeline([
        ("extractor", SequenceTensorExtractor(
            acc_modes="smoothed|velocity|jerk",
            rotation_modes="quaternion|angular_velocity",
            tof_modes="pooled_stats",
            thm_modes="centered_diff",
            sampling_rate=50,
            maxlen=150,
            window_size=40,
            clip_value=50.0,
            interp_mode="linear",
            motion_filter_mode="kalman",
            use_dead_reckoning=False,
            dead_reckoning_detrend=False,
            kalman_process_noise=1e-3,
            kalman_measurement_noise=1e-1
        )),
        ("classifier", HierarchicalBFRBEnsemble(
            orientation_col=orientation_col,
            target_col="bfrb", # Ensemble is strictly designed for BFRB
            l3_params_dict=l3_override_dict,
            random_state=random_state
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        ensemble_search = BayesSearchCV(
            ensemble_pipe, ensemble_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        ensemble_search = GridSearchCV(
            ensemble_pipe, ensemble_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    ensemble_search.fit(X_train, y_train, groups=groups)
    fitted_models["Hierarchical Ensemble"] = ensemble_search.best_estimator_
    
    y_pred_ens = ensemble_search.predict(X_test)
    
    eval_dict_ens = evaluate_holdout(y_test, y_pred_ens, target_col="bfrb", verbose=True)
    score_ens = eval_dict_ens.get("competition_score", eval_dict_ens.get("holdout_score", 0))
    
    print(f"Hierarchical Ensemble Best CV Score: {ensemble_search.best_score_:.4f} | Holdout Score: {score_ens:.4f}")
    print(f"Best Params: {ensemble_search.best_params_}")
    
    results_list.append({
        "Model": "Hierarchical Ensemble",
        "Target": "bfrb",
        "CV Score": ensemble_search.best_score_,
        "Holdout Score": score_ens,
        "Best Params": ensemble_search.best_params_,
    })

# ============================================================
# FINAL SUMMARY
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "="*60)
print("BASELINES SUMMARY")
print("="*60)
print(results_df.to_string(index=False))


--- Training: Single MiniRocket (Target: bfrb) ---
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END classifier__alpha=1000.0, classifier__class_weight=balanced, classifier__feature_selection_percentile=None, classifier__num_kernels=840, classifier__target_col=bfrb, extractor__acc_modes=raw, extractor__clip_value=None, extractor__dead_reckoning_detrend=False, extractor__interp_mode=linear, extractor__kalman_measurement_noise=0.1, extractor__kalman_process_noise=0.001, extractor__maxlen=180, extractor__motion_filter_mode=None, extractor__padding_value=0.0, extractor__rotation_modes=raw, extractor__sampling_rate=20, extractor__thm_modes=raw, extractor__tof_modes=raw, extractor__use_dead_reckoning=False, extractor__window_size=20;, score=(train=0.991, test=0.542) total time=  53.3s

FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.9242
BFRB Gesture Macro F1: 0.3454
COMPETITION SCORE: 0.6348

----------------------------------------
BFRB Gesture Classification Report

In [26]:
# ============================================================
# FINAL SUMMARY
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "="*60)
print("BASELINES SUMMARY")
print("="*60)
print(results_df.to_string(index=False))

if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"]
    print(f"\nDetailed holdout eval for best model: {best_name}")
    best_model = fitted_models[best_name]
    
    y_pred = best_model.predict(X_test)
    
    # Collapse y_test to sequence level to match the length of y_pred
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
    
    print("\nClassification Report:")
    print(classification_report(y_test_seq[TARGET_COL], y_pred, zero_division=0))


BASELINES SUMMARY
            Model Target  CV Score  Holdout Score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             Best Params
Single MiniRocket   bfrb  0.541636       0.634778 {'classifier__alpha': 1000.0, 'classifier__class_weight': 'balanced', 'classifier__feature_selection_percentile': None, 'classifier__num_kernels': 840, 'classifier__targ

NameError: name 'target_col' is not defined

In [ ]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit

from sklearn.metrics import f1_score, classification_report

import data_utils
from base_utils_qwen import (
    SequenceExtractor,
    prepare_multitask_param_space,
    prepare_bayesian_space,
    competition_scorer as bfrb_competition_scorer,
    evaluate_holdout,
    plot_training_curves,
)
from proto_utils_qwen import SingleHeadPrototypicalNetwork, MultiHeadPrototypicalNetwork
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, make_scorer

In [ ]:
# ============================================================
# CONFIGURATION - CHANGE THIS TO SWITCH MODES
# ============================================================

# Set this to "single" or "multi"
mode = "multi"  # "single" for binary+gesture, "multi" for multi-head

single_target = "bfrb"  # For single mode: "gesture", "gesture_action", etc.

# For MULTI mode: which heads to predict
multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "bfrb"  # For multi mode: what to evaluate F1 on
target_col = primary_target

pipe_name = "extractor"
proto_name = "model"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
n_jobs = 1
train_size = 0.5
error_score_constant = 0.0
verbose = 1
do_cross_val = False

filter_non_brb_classes = False
filter_orientation_class_list = None

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)  # ✅ Fast

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if target_col == 'bfrb':
    competition_scorer = bfrb_competition_scorer
else:
    competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

print(f"Mode: {mode}")
print(f"Search mode: {search_mode}")

In [ ]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

In [ ]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
train_df = train_df.drop(columns=["handedness"])

In [ ]:
# Create target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

if filter_non_brb_classes:
    train_df = train_df.loc[train_df['is_target'],:]

if filter_orientation_class_list is not None:
    train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]

    # Get unique sequences
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()

    # Stratified split by your target column
    train_seqs, test_seqs = train_test_split(
        sequences['sequence_id'], 
        test_size=(1 - train_size), 
        stratify=sequences[target_col],  # or use multiple columns
        random_state=random_state
    )

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]
else:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
    )


X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

In [ ]:
train_sample_df.groupby([target_col, 'orientation']).agg(
    {'sequence_id':'nunique'})

In [ ]:
sequence_extractor = SequenceExtractor(
    acc_modes='raw'
)

if mode == "single":
    # BinaryPlusGesturePrototypicalNetwork uses is_target and gesture automatically
    model = SingleHeadPrototypicalNetwork(
        target=single_target,
    )
else:
    model = MultiHeadPrototypicalNetwork(
        primary_target=primary_target,
    )

pipeline = Pipeline([
    ("extractor", sequence_extractor),
    ("model", model),
])

In [ ]:
# ============================================================================
# PARAMETER SPACE DEFINITION
# ============================================================================
if search_mode == "bayesian":
    base_extractor_params = {
        f"{pipe_name}__acc_modes": Categorical(["smoothed|velocity|jerk", "raw"]),
        f"{pipe_name}__rotation_modes": Categorical(["quaternion|euler|angular_velocity", "quaternion|delta_euler"]),
        f"{pipe_name}__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"]),
        f"{pipe_name}__thm_modes": Categorical(["centered_diff", "diff"]),
        f"{pipe_name}__motion_filter_mode": Categorical(["kalman", "extended_kalman", None]),
        f"{pipe_name}__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-2, 1e-1, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True, False]),
        f"{pipe_name}__dead_reckoning_detrend": Categorical([True, False]),
        # CRITICAL: Rank 1 used 20Hz! Lower sampling acts as a powerful regularizer.
        f"{pipe_name}__sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0, 100.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear", "ffill"]),
        f"{pipe_name}__window_size": Integer(10, 50),
        f"{pipe_name}__smooth_alpha": Real(0.1, 0.9),
        f"{pipe_name}__maxlen": Categorical([120, 160, 200]),
        f"{pipe_name}__padding_value": Categorical([-999.0]),
    }

    base_model_params = {
        f"{proto_name}__backbone_type": Categorical(["1dcnn"]),
        f"{proto_name}__filters": Categorical(["64-128-256", "32-64-128", "64-128"]),
        f"{proto_name}__kernels": Categorical(["5-3", "7-5-3", "3-3"]),
        f"{proto_name}__pools": Categorical(["none", "2", "2-2"]),

        # Regularization narrowed around the ~0.45 sweet spot
        f"{proto_name}__dropout": Real(0.35, 0.55),
        f"{proto_name}__spatial_dropout": Real(0.1, 0.3),  # NEW
        f"{proto_name}__l2_reg": Real(1e-5, 1e-3, prior="log-uniform"),  # NEW
        f"{proto_name}__temperature": Real(0.05, 0.2, prior="log-uniform"),  # NEW
        f"{proto_name}__embed_dim": Categorical([64, 128]),

        # Learning rate narrowed around the 3.5e-3 winner
        f"{proto_name}__learning_rate": Real(1e-3, 5e-3, prior="log-uniform"),
        f"{proto_name}__batch_size": Categorical([16, 32]),
        f"{proto_name}__epochs": Categorical([50]),
        f"{proto_name}__patience": Categorical([10, 15]),

        f"{proto_name}__n_way": Categorical([9]),
        f"{proto_name}__n_support": Categorical([20, 40, 60]),
        f"{proto_name}__n_query": Categorical([20, 40, 60]),

        f"{proto_name}__use_mixup": Categorical([True, False]),
        f"{proto_name}__mixup_alpha": Real(0.2, 0.5),
        f"{proto_name}__use_time_mask": Categorical([True]),
        f"{proto_name}__time_mask_ratio": Real(0.10, 0.20),
        f"{proto_name}__use_gaussian_noise": Categorical([True]),
        f"{proto_name}__noise_std": Real(0.001, 0.01, prior="log-uniform"),
    }

    if mode == "single":
        param_space = {**base_extractor_params, **base_model_params, f"{proto_name}__target": Categorical([single_target])}
    else:
        heads_tuple = tuple(multi_heads)
        param_space = {**base_extractor_params, **base_model_params, f"{proto_name}__sub_heads": Categorical([heads_tuple]), f"{proto_name}__primary_target": Categorical([primary_target])}

    param_space = prepare_bayesian_space(param_space)

else:  # Grid Search (Tightly constrained to proven winners)
    param_space = {
        f"{pipe_name}__acc_modes": ["smoothed|velocity|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|euler|angular_velocity"],
        f"{pipe_name}__tof_modes": ["sensor_stats"],
        f"{pipe_name}__thm_modes": ["centered_diff"],
        f"{pipe_name}__motion_filter_mode": ["kalman"],
        f"{pipe_name}__sampling_rate": [20],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__window_size": [50],
        f"{pipe_name}__maxlen": [200],
        f"{pipe_name}__padding_value": [-999.0],

        f"{proto_name}__backbone_type": ["1dcnn"],
        f"{proto_name}__filters": ["128-128-256"],
        f"{proto_name}__kernels": ["5-3"],
        f"{proto_name}__pools": ["2"],

        f"{proto_name}__dropout": [0.45],
        f"{proto_name}__spatial_dropout": [0.15],  # NEW
        f"{proto_name}__l2_reg": [1e-4],  # NEW
        f"{proto_name}__temperature": [0.1],  # NEW
        f"{proto_name}__embed_dim": [128],
        f"{proto_name}__lstm_units": [128],

        f"{proto_name}__learning_rate": [5e-3],
        f"{proto_name}__batch_size": [16],
        f"{proto_name}__epochs": [50],
        f"{proto_name}__patience": [15],

        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [40],
        f"{proto_name}__n_query": [40],

        f"{proto_name}__use_mixup": [False],
        f"{proto_name}__use_time_mask": [True],
        f"{proto_name}__time_mask_ratio": [0.15],
        f"{proto_name}__use_gaussian_noise": [True],
        f"{proto_name}__noise_std": [0.005],
    }

    if mode == "single":
        param_space[f"{proto_name}__target"] = [single_target]
    else:
        param_space[f"{proto_name}__primary_target"] = [primary_target]
        param_space[f"{proto_name}__sub_heads"] = [tuple(multi_heads)]


In [ ]:
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    # For multi mode, include all heads plus the primary target for lookup
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols:
        cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", primary_target]].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
groups = X_train["sequence_id"]

print(f"MODE: {mode}")
print(f"Single target: {single_target if mode=='single' else 'N/A'}")
print(f"Multi heads: {multi_heads if mode=='multi' else 'N/A'}")
print(f"Primary target: {primary_target if mode=='multi' else 'N/A'}")

print(f"y_train columns: {y_train.columns.tolist()}")

In [ ]:
if search_mode == "bayesian":

    param_space = prepare_multitask_param_space(param_space, search_mode)
    
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=competition_scorer,
        cv=cv_object,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=competition_scorer,
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

In [ ]:
# ============================================================
# SAVE CV RESULTS TO CSV
# ============================================================

# Convert search results to DataFrame
cv_results_df = pd.DataFrame(search.cv_results_)

# Add metadata columns
cv_results_df['mode'] = mode
cv_results_df['search_mode'] = search_mode
cv_results_df['timestamp'] = timestamp

# Define save path
results_path = results_dir / f"cv_results_{mode}_{search_mode}_{timestamp}.csv"

# Save to CSV
cv_results_df.to_csv(results_path, index=False)

print(f"CV results saved to: {results_path}")
print(f"Shape: {cv_results_df.shape}")
print(f"Columns: {cv_results_df.columns.tolist()}")

In [ ]:
# ============================================================
# EVALUATE ON HOLDOUT TEST SET - WORKS FOR BOTH MODES
# ============================================================
y_test_true = hold_out_df[["sequence_id", "is_target", target_col]].copy()
y_pred = best_model.predict(X_test)

holdout_eval = evaluate_holdout(y_test_true, y_pred, target_col=primary_target)
competition_score = holdout_eval["competition_score"]

holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_eval["results_df"].to_csv(holdout_results_path, index=False)
print(f"\nHoldout predictions saved to: {holdout_results_path}")

In [ ]:
# ============================================================
# TRAINING / VALIDATION LOSS & ACCURACY CURVES
# ============================================================
proto_clf = best_model.named_steps[proto_name]
if hasattr(proto_clf, "history_") and proto_clf.history_:
    plot_training_curves(
        proto_clf.history_,
        title=f"Prototypical Network ({mode} mode)",
    )
    plt.show()
else:
    print("No training history found on best model — history_ is populated after .fit()")

# MultiRocket Experiments Notebook

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score, make_scorer
from sklearn.linear_model import RidgeClassifier, LogisticRegression
import itertools

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# Ensure sktime and skopt are available
try:
    import sktime
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "sktime", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import data_utils
from base_utils_qwen import (
    SequenceExtractor,
    competition_scorer,
    evaluate_holdout,
    make_competition_scorer,
    prepare_bayesian_space
)
from multi_rocket_utils import (
    FlexibleMultiRocketClassifier,
    HierarchicalMultiRocketEnsemble
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 5.7 MB/s eta 0:00:00


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
MODE = "single"  # "ensemble" or "single"
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "grid"  # "grid" or "bayesian"

random_state = 42
n_splits = 3
n_iter = 25
train_size = 0.4 # Using 80% for training now instead of small balanced split
error_score_constant = 0.0
verbose = 3
do_cross_val = False

# Slicing options for Single Mode
slice_by_orientation = None  # e.g., ["Seated Straight", "Standing"]
slice_by_bfrb = None         # True for BFRB only, False for non-BFRB only, None for all

do_handness = False

# Ensemble orientations split (Leave None to auto-split evenly)
l3_orientations = None 
l4_orientations = None 

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Scorer selection based on target
if target_col == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

print(f"MODE: {MODE}")
print(f"Search mode: {search_mode}")

MODE: single
Search mode: grid


In [3]:
data_root = data_utils.find_data_root()
raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness:
# Handedness & Upside-down corrections
    if "handedness" in train_demo_df.columns:
        train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
        left_handed_mask = train_df["handedness"].eq(0)
        train_df.loc[left_handed_mask, "acc_x"] *= -1.0
        train_df = train_df.drop(columns=["handedness"])

    upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
    train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
    train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Create alternative target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [4]:
import itertools
from base_utils_qwen import SequenceExtractor

# Take a small sample of X_train for fast computation
sample_df = X_train.head(2000)

# Extract extractor parameter names and values from param_space
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]

# Check if we're in Bayesian mode (contains skopt objects) or Grid mode (lists)
is_bayesian = search_mode == "bayesian" and SKOPT_AVAILABLE

if is_bayesian:
    print("Bayesian mode detected - sampling 5 random combinations for channel calculation\n")
    import random
    from skopt.space import Categorical, Integer, Real
    
    # For Bayesian, we'll sample random combinations
    n_samples = 5
    
    for i in range(n_samples):
        extractor_params = {}
        for key in extractor_keys:
            param_name = key.replace("extractor__", "")
            space = param_space[key]
            
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                if space.prior == "log-uniform":
                    value = np.exp(random.uniform(np.log(space.low), np.log(space.high)))
                else:
                    value = random.uniform(space.low, space.high)
            else:
                value = space
            
            extractor_params[param_name] = value
        
        # Ensure padding_value=0.0
        extractor_params["padding_value"] = 0.0
        
        try:
            extractor = SequenceExtractor(**extractor_params)
            extractor.fit(sample_df)
            out = extractor.transform(sample_df)
            n_channels = out['X'].shape[2]
            print(f"Sample {i+1}:")
            # Show only key params
            key_params = {k: v for k, v in extractor_params.items() if k in ['acc_modes', 'rotation_modes', 'sampling_rate', 'maxlen', 'window_size']}
            print(f"  Params: {key_params}")
            print(f"  → Channels: {n_channels}\n")
        except Exception as e:
            print(f"Failed for {extractor_params}: {e}\n")

else:
    # Grid mode - iterate over all combinations
    extractor_values = [param_space[k] for k in extractor_keys]
    total_combos = len(list(itertools.product(*extractor_values)))
    print(f"Total extractor parameter combinations: {total_combos}\n")
    
    for combo in itertools.product(*extractor_values):
        extractor_params = {k.replace("extractor__", ""): v for k, v in zip(extractor_keys, combo)}
        
        # Ensure padding_value=0.0 (required for MultiRocket)
        extractor_params["padding_value"] = 0.0
        
        try:
            extractor = SequenceExtractor(**extractor_params)
            extractor.fit(sample_df)
            out = extractor.transform(sample_df)
            n_channels = out['X'].shape[2]
            # Show only key params to avoid clutter
            key_params = {k: v for k, v in extractor_params.items() if k in ['acc_modes', 'rotation_modes', 'sampling_rate', 'maxlen', 'window_size']}
            print(f"Params: {key_params}")
            print(f"→ Channels: {n_channels}\n")
        except Exception as e:
            print(f"Failed for {extractor_params}: {e}\n")

Train sequences: 3260 | Test sequences: 4891


In [5]:
# ============================================================
# PARAMETER SPACE DEFINITION
# ============================================================
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    extractor_space = {
        "extractor__acc_modes": Categorical(["raw"]),
        "extractor__rotation_modes": Categorical(['quaternion|angular_velocity|delta_euler|rot6d']),
        "extractor__tof_modes": Categorical(['pooled_stats|sensor_stats']),
        "extractor__thm_modes": Categorical(['centered_diff']),
        "extractor__sampling_rate": Categorical([20]), 
        "extractor__maxlen": Categorical([160]),
        "extractor__window_size": Categorical([5]),
        "extractor__clip_value": Categorical([None]),
        "extractor__interp_mode": Categorical(["linear"]),
        "extractor__motion_filter_mode": Categorical([None]),
        "extractor__kalman_process_noise": Categorical([1e-4]),
        "extractor__kalman_measurement_noise": Categorical([1e-2]),
        "extractor__padding_value": Categorical([0.0]), 
    }

    # TRUE MULTIPLES OF 84 (Aeon MultiRocket requirement)
    kernels_space = Categorical([84*2, 84*10, 84*60])
    max_dilations_space = Integer(2, 24)
    pad_space = Categorical([True])
    use_pyfftw_space = Categorical([False, True])
    use_multivariate_space = Categorical([False, True])
    # alpha_space = Real(1e1, 1.5e3, prior="log-uniform")
    alpha_space = Categorical([1e1, 1.67e3, 3.7e3])
    fs_space = Categorical([1, 5, 10, 50])
    channels_space = Categorical([None, 10, 20, 40])

    if MODE == "ensemble":
        model_space = {
            "classifier__base_classifier_class": Categorical([RidgeClassifier]),
            
            # Layer 1
            "classifier__l1_num_kernels": kernels_space,
            "classifier__l1_max_dilations_per_kernel": max_dilations_space,
            "classifier__l1_pad": pad_space,
            "classifier__l1_use_pyfftw": use_pyfftw_space,
            "classifier__l1_use_multivariate": use_multivariate_space,
            "classifier__l1_alpha": alpha_space, 
            "classifier__l1_feature_selection_percentile": fs_space,
            "classifier__l1_max_channels": channels_space,
            "classifier__l1_class_weight": Categorical([None]),
            
            # Layer 2
            "classifier__l2_num_kernels": kernels_space,
            "classifier__l2_max_dilations_per_kernel": max_dilations_space,
            "classifier__l2_pad": pad_space,
            "classifier__l2_use_pyfftw": use_pyfftw_space,
            "classifier__l2_use_multivariate": use_multivariate_space,
            "classifier__l2_alpha": alpha_space, 
            "classifier__l2_feature_selection_percentile": fs_space,
            "classifier__l2_max_channels": channels_space,
            "classifier__l2_class_weight": Categorical([None]),
            
            # Layer 3 Model 1
            "classifier__l3_1_num_kernels": kernels_space,
            "classifier__l3_1_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_1_pad": pad_space,
            "classifier__l3_1_use_pyfftw": use_pyfftw_space,
            "classifier__l3_1_use_multivariate": use_multivariate_space,
            "classifier__l3_1_alpha": alpha_space,
            "classifier__l3_1_feature_selection_percentile": fs_space,
            "classifier__l3_1_max_channels": channels_space,
            
            # Layer 3 Model 2
            "classifier__l3_2_num_kernels": kernels_space,
            "classifier__l3_2_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_2_pad": pad_space,
            "classifier__l3_2_use_pyfftw": use_pyfftw_space,
            "classifier__l3_2_use_multivariate": use_multivariate_space,
            "classifier__l3_2_alpha": alpha_space,
            "classifier__l3_2_feature_selection_percentile": fs_space,
            "classifier__l3_2_max_channels": channels_space,
            
            # Layer 3 Model 3
            "classifier__l3_3_num_kernels": kernels_space,
            "classifier__l3_3_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_3_pad": pad_space,
            "classifier__l3_3_use_pyfftw": use_pyfftw_space,
            "classifier__l3_3_use_multivariate": use_multivariate_space,
            "classifier__l3_3_alpha": alpha_space,
            "classifier__l3_3_feature_selection_percentile": fs_space,
            "classifier__l3_3_max_channels": channels_space,
            
            # Layer 3 Model 4
            "classifier__l3_4_num_kernels": kernels_space,
            "classifier__l3_4_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_4_pad": pad_space,
            "classifier__l3_4_use_pyfftw": use_pyfftw_space,
            "classifier__l3_4_use_multivariate": use_multivariate_space,
            "classifier__l3_4_alpha": alpha_space,
            "classifier__l3_4_feature_selection_percentile": fs_space,
            "classifier__l3_4_max_channels": channels_space,
            
            "classifier__l3_class_weight": Categorical([None]),
        }
        fixed_model_params = {
            "classifier__orientation_col": [orientation_col],
            "classifier__target_col": [target_col],
            "classifier__padding_value": [0.0],
        }
    else:  # single mode
        model_space = {
            "classifier__base_classifier_class": Categorical([RidgeClassifier]),
            "classifier__num_kernels": kernels_space,
            "classifier__max_dilations_per_kernel": max_dilations_space,
            "classifier__pad": pad_space,
            "classifier__use_pyfftw": use_pyfftw_space,
            "classifier__use_multivariate": use_multivariate_space,
            "classifier__alpha": alpha_space,
            "classifier__feature_selection_percentile": fs_space,
            "classifier__max_channels": channels_space,
        }
        fixed_model_params = {
            "classifier__target_col": [target_col],
            "classifier__orientation_col": [orientation_col],
            "classifier__slice_by_orientation": [slice_by_orientation],
            "classifier__slice_by_bfrb": [slice_by_bfrb],
            "classifier__padding_value": [0.0],
            "classifier__class_weight": Categorical(["balanced", None]),
        }

    param_space = {**extractor_space, **model_space, **fixed_model_params}
    param_space = prepare_bayesian_space(param_space)

else:  # GRID SEARCH
    param_space = {
        "extractor__acc_modes": ['smoothed|velocity|displacement|jerk'],
        "extractor__rotation_modes": ['quaternion|euler|angular_velocity'],
        "extractor__tof_modes": ['pooled_stats|sensor_stats'],
        "extractor__thm_modes": ['centered'],
        "extractor__sampling_rate": [200],
        "extractor__maxlen": [200],
        "extractor__window_size": [5],
        "extractor__clip_value": [None],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": [None],
        "extractor__kalman_process_noise": [1e-3],
        "extractor__kalman_measurement_noise": [1e-1],
        "extractor__padding_value": [0.0],
    }
    
    # MultiRocket classifier hyperparameters for grid search
    kernals_list = [84*60]
    max_dilations_list = [16]
    pad_list = [False]
    use_pyfftw_list = [False]  # Keep False unless you have FFTW installed
    use_multivariate_list = [False]
    alphas_list = [3.7e2, 1.37e3]
    feature_selection_list = [1, 10]
    max_channels_list = [50]
    classifier_padding_value_list = [0.0]
    class_weight_list = ['balanced']

    if MODE == "ensemble":
        param_space.update({
            "classifier__base_classifier_class": [RidgeClassifier],
            # Layer 1
            "classifier__l1_num_kernels": kernals_list,
            "classifier__l1_max_dilations_per_kernel": max_dilations_list,
            "classifier__l1_pad": pad_list,
            "classifier__l1_use_pyfftw": use_pyfftw_list,
            "classifier__l1_use_multivariate": use_multivariate_list,
            "classifier__l1_alpha": alphas_list,
            "classifier__l1_feature_selection_percentile": feature_selection_list,
            "classifier__l1_max_channels": max_channels_list,
            "classifier__l1_class_weight": class_weight_list,
            # Layer 2
            "classifier__l2_num_kernels": kernals_list,
            "classifier__l2_max_dilations_per_kernel": max_dilations_list,
            "classifier__l2_pad": pad_list,
            "classifier__l2_use_pyfftw": use_pyfftw_list,
            "classifier__l2_use_multivariate": use_multivariate_list,
            "classifier__l2_alpha": alphas_list,
            "classifier__l2_feature_selection_percentile": feature_selection_list,
            "classifier__l2_max_channels": max_channels_list,
            "classifier__l2_class_weight": class_weight_list,
            # Layer 3 Model 1
            "classifier__l3_1_num_kernels": kernals_list,
            "classifier__l3_1_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_1_pad": pad_list,
            "classifier__l3_1_use_pyfftw": use_pyfftw_list,
            "classifier__l3_1_use_multivariate": use_multivariate_list,
            "classifier__l3_1_alpha": alphas_list,
            "classifier__l3_1_feature_selection_percentile": feature_selection_list,
            "classifier__l3_1_max_channels": max_channels_list,
            # Layer 3 Model 2
            "classifier__l3_2_num_kernels": kernals_list,
            "classifier__l3_2_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_2_pad": pad_list,
            "classifier__l3_2_use_pyfftw": use_pyfftw_list,
            "classifier__l3_2_use_multivariate": use_multivariate_list,
            "classifier__l3_2_alpha": alphas_list,
            "classifier__l3_2_feature_selection_percentile": feature_selection_list,
            "classifier__l3_2_max_channels": max_channels_list,
            # Layer 3 Model 3
            "classifier__l3_3_num_kernels": kernals_list,
            "classifier__l3_3_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_3_pad": pad_list,
            "classifier__l3_3_use_pyfftw": use_pyfftw_list,
            "classifier__l3_3_use_multivariate": use_multivariate_list,
            "classifier__l3_3_alpha": alphas_list,
            "classifier__l3_3_feature_selection_percentile": feature_selection_list,
            "classifier__l3_3_max_channels": max_channels_list,
            # Layer 3 Model 4
            "classifier__l3_4_num_kernels": kernals_list,
            "classifier__l3_4_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_4_pad": pad_list,
            "classifier__l3_4_use_pyfftw": use_pyfftw_list,
            "classifier__l3_4_use_multivariate": use_multivariate_list,
            "classifier__l3_4_alpha": alphas_list,
            "classifier__l3_4_feature_selection_percentile": feature_selection_list,
            "classifier__l3_4_max_channels": max_channels_list,
            "classifier__l3_class_weight": class_weight_list,
            "classifier__orientation_col": [orientation_col],
            "classifier__target_col": [target_col],
            "classifier__padding_value": classifier_padding_value_list,
        })
    else:  # single mode
        param_space.update({
            "classifier__base_classifier_class": [RidgeClassifier],
            "classifier__num_kernels": kernals_list,
            "classifier__max_dilations_per_kernel": max_dilations_list,
            "classifier__pad": pad_list,
            "classifier__use_pyfftw": use_pyfftw_list,
            "classifier__use_multivariate": use_multivariate_list,
            "classifier__alpha": alphas_list,
            "classifier__feature_selection_percentile": feature_selection_list,
            "classifier__max_channels": max_channels_list,
            "classifier__target_col": [target_col],
            "classifier__orientation_col": [orientation_col],
            "classifier__slice_by_orientation": [slice_by_orientation],
            "classifier__slice_by_bfrb": [slice_by_bfrb],
            "classifier__padding_value": classifier_padding_value_list,
            "classifier__class_weight": class_weight_list,
        })

print("Parameter Space Defined.")

Parameter Space Defined.


In [6]:
import itertools
import random
import numpy as np
from base_utils_qwen import SequenceExtractor
from multi_rocket_utils import FlexibleMultiRocketClassifier

# Take a small sample of X_train for fast computation
sample_df = X_train.head(2000)

# Extract extractor parameter names and values from param_space
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]
classifier_keys = [k for k in param_space.keys() if k.startswith("classifier__")]

# Check if we're in Bayesian mode
is_bayesian = search_mode == "bayesian" and SKOPT_AVAILABLE

n_samples = 10

print(f"--- Estimating MultiRocket Output Features ({'Bayesian' if is_bayesian else 'Grid'} Mode) ---\n")

for i in range(n_samples):
    # Sample extractor params
    extractor_params = {}
    for key in extractor_keys:
        param_name = key.replace("extractor__", "")
        space = param_space[key]
        
        if is_bayesian:
            from skopt.space import Categorical, Integer, Real
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                if space.prior == "log-uniform":
                    value = np.exp(random.uniform(np.log(space.low), np.log(space.high)))
                else:
                    value = random.uniform(space.low, space.high)
            else:
                value = space
        else:
            value = random.choice(space) if isinstance(space, list) else space
        
        extractor_params[param_name] = value
    
    # Sample classifier params (for MultiRocket)
    classifier_params = {}
    for key in classifier_keys:
        param_name = key.replace("classifier__", "")
        space = param_space[key]
        
        if is_bayesian:
            from skopt.space import Categorical, Integer, Real
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                if space.prior == "log-uniform":
                    value = np.exp(random.uniform(np.log(space.low), np.log(space.high)))
                else:
                    value = random.uniform(space.low, space.high)
            else:
                value = space
        else:
            value = random.choice(space) if isinstance(space, list) else space
        
        classifier_params[param_name] = value
    
    # Ensure padding_value=0.0
    extractor_params["padding_value"] = 0.0
    classifier_params["padding_value"] = 0.0
    
    try:
        # Step 1: Extract features to get input channels
        extractor = SequenceExtractor(**extractor_params)
        extractor.fit(sample_df)
        out = extractor.transform(sample_df)
        
        if isinstance(out, dict):
            X_arr = out['X']
        else:
            X_arr = out
        
        n_input_channels = X_arr.shape[-1]
        
        # Step 2: Get MultiRocket hyperparameters
        num_kernels = classifier_params.get('num_kernels', 168)
        max_dilations = classifier_params.get('max_dilations_per_kernel', 32)
        use_multivariate = classifier_params.get('use_multivariate', True)
        
        # Step 3: Calculate MultiRocket output features
        # For multivariate: features = num_kernels × max_dilations_per_kernel
        # (actually slightly more due to bias features, but this is approximate)
        if use_multivariate:
            rocket_features = num_kernels * max_dilations
        else:
            rocket_features = num_kernels * max_dilations * n_input_channels
        
        # With feature selection
        fs_percentile = classifier_params.get('feature_selection_percentile', None)
        if fs_percentile is not None and fs_percentile < 100:
            final_features = int(rocket_features * fs_percentile / 100)
        else:
            final_features = rocket_features
        
        # Show only key params
        key_extractor = {k: v for k, v in extractor_params.items() 
                        if k in ['acc_modes', 'rotation_modes', 'tof_modes', 'thm_modes', 
                                 'sampling_rate', 'maxlen']}
        
        print(f"Sample {i+1}:")
        print(f"  Extractor: {key_extractor}")
        print(f"  Input channels: {n_input_channels}")
        print(f"  MultiRocket: num_kernels={num_kernels}, max_dilations={max_dilations}")
        print(f"  Rocket features (before selection): {rocket_features:,}")
        print(f"  Feature selection: {fs_percentile}% → {final_features:,} features")
        print(f"  Total output features to classifier: {final_features:,}\n")
        
    except Exception as e:
        print(f"Failed for sample {i+1}: {e}\n")

print("--- Estimation Complete ---")

--- Estimating MultiRocket Output Features (Grid Mode) ---

Sample 1:
  Extractor: {'acc_modes': 'smoothed|velocity|displacement|jerk', 'rotation_modes': 'quaternion|euler|angular_velocity', 'tof_modes': 'pooled_stats|sensor_stats', 'thm_modes': 'centered', 'sampling_rate': 200, 'maxlen': 200}
  Input channels: 153
  MultiRocket: num_kernels=5040, max_dilations=16
  Rocket features (before selection): 12,337,920
  Feature selection: 1% → 123,379 features
  Total output features to classifier: 123,379

Sample 2:
  Extractor: {'acc_modes': 'smoothed|velocity|displacement|jerk', 'rotation_modes': 'quaternion|euler|angular_velocity', 'tof_modes': 'pooled_stats|sensor_stats', 'thm_modes': 'centered', 'sampling_rate': 200, 'maxlen': 200}
  Input channels: 153
  MultiRocket: num_kernels=5040, max_dilations=16
  Rocket features (before selection): 12,337,920
  Feature selection: 1% → 123,379 features
  Total output features to classifier: 123,379

Sample 3:
  Extractor: {'acc_modes': 'smoothed

In [7]:
pipe = Pipeline([
    # We override padding_value to 0.0 here because MultiRocket convolutions react poorly to -999.0
    ("extractor", SequenceExtractor(padding_value=0.0)), 
    ("classifier", HierarchicalMultiRocketEnsemble(padding_value=0.0) if MODE == "ensemble" else FlexibleMultiRocketClassifier(padding_value=0.0))
])

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        pipe, param_space, n_iter=n_iter, scoring=scorer,
        cv=cv_object, n_jobs=1, random_state=random_state,
        error_score=error_score_constant, verbose=verbose, return_train_score=True
    )
else:
    search = GridSearchCV(
        pipe, param_space, scoring=scorer,
        cv=cv_object, n_jobs=1, error_score=error_score_constant, 
        verbose=verbose, return_train_score=True
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

Running grid search...
Fitting 1 folds for each of 4 candidates, totalling 4 fits
[CV 1/1] END classifier__alpha=370.0, classifier__base_classifier_class=<class 'sklearn.linear_model._ridge.RidgeClassifier'>, classifier__class_weight=balanced, classifier__feature_selection_percentile=1, classifier__max_channels=50, classifier__max_dilations_per_kernel=16, classifier__num_kernels=5040, classifier__orientation_col=orientation, classifier__pad=False, classifier__padding_value=0.0, classifier__slice_by_bfrb=None, classifier__slice_by_orientation=None, classifier__target_col=bfrb, classifier__use_multivariate=False, classifier__use_pyfftw=False, extractor__acc_modes=smoothed|velocity|displacement|jerk, extractor__clip_value=None, extractor__interp_mode=linear, extractor__kalman_measurement_noise=0.1, extractor__kalman_process_noise=0.001, extractor__maxlen=200, extractor__motion_filter_mode=None, extractor__padding_value=0.0, extractor__rotation_modes=quaternion|euler|angular_velocity, extr

In [8]:
if not do_cross_val and len(X_test) > 0:
    y_pred = best_model.predict(X_test)
    
    # Collapse to sequence level
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
    
    if target_col == 'bfrb':
        eval_dict = evaluate_holdout(y_test_seq, y_pred, target_col=target_col, verbose=True)
        print(f"Holdout Competition Score: {eval_dict['competition_score']:.4f}")
    else:
        print("\nClassification Report:")
        print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))
        macro_f1 = f1_score(y_test_seq[target_col], y_pred, average='macro', zero_division=0)
        print(f"Macro F1 Score: {macro_f1:.4f}")
else:
    print("Holdout evaluation skipped (Cross-Val mode or empty test set).")


FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.8772
BFRB Gesture Macro F1: 0.3365
COMPETITION SCORE: 0.6069

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.50      0.51      0.50       396
      Cheek - pinch skin       0.45      0.26      0.33       386
     Eyebrow - pull hair       0.33      0.13      0.18       364
     Eyelash - pull hair       0.32      0.30      0.31       370
Forehead - pull hairline       0.42      0.35      0.38       370
      Forehead - scratch       0.42      0.66      0.51       391
       Neck - pinch skin       0.40      0.35      0.37       382
          Neck - scratch       0.44      0.43      0.44       393
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.38      3052
               macro avg       0.36      0.

 # V4 Prototypical Network (DEFINITIVE & EXHAUSTIVE)

 This notebook integrates EVERY parameter, augmentation, backbone, and preprocessing step.

In [1]:
import os, sys, warnings, logging, numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit, train_test_split
from sklearn.metrics import f1_score, make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)
src_path = os.path.join(workspace_root, "src")
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

try:
    import data_utils
    from base_utils_qwen import prepare_bayesian_space, competition_scorer as bfrb_competition_scorer, evaluate_holdout, SequenceExtractor
    from proto_utils_v4 import V4PrototypicalNetwork, V4MultiHeadPrototypicalNetwork
    print("✅ Imports loaded successfully")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise


✅ Imports loaded successfully


 # 1. Global Configuration & Slicing Flags

In [2]:
mode = "single"  # "single" or "multi"
single_target = "bfrb" 
target_col = single_target

multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "bfrb" 

pipe_name = "extractor"
proto_name = "model"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 30  
n_jobs = 1
train_size = 0.5
error_score_constant = 'raise'
verbose = 3
do_cross_val = False

if do_cross_val: cv_object = GroupKFold(n_splits=n_splits)
else: cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

if target_col == 'bfrb': competition_scorer = bfrb_competition_scorer
else: competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# DATA FILTERING & SLICING FLAGS (From siamese/prototypical_v2)
use_eg_sample = False
eg_sample_pct = 0.02
do_handness = False                 
do_upside_down = False
filter_non_bfrb_classes = False    
filter_orientation_class_list = None 
slice_by_orientation = None        
slice_by_bfrb = None               

print(f"Mode: {mode} | Search: {search_mode} | Target: {target_col}")


Mode: single | Search: grid | Target: bfrb


 # 2. Data Loading & Exhaustive Preprocessing

In [3]:
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(frac=eg_sample_pct, random_state=random_state)
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness and "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
    q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)
    q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)
    norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
    bad_q = norm.squeeze() == 0
    q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])
    norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
    q_wxyz = q_wxyz / norm
    q_xyzw = q_wxyz[:, [1, 2, 3, 0]]
    euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)
    euler_xyz[:, [1, 2]] *= -1.0
    q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()
    q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]
    train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed
    train_df = train_df.drop(columns=["handedness"])

if do_upside_down:
    upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
    train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
    train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

if filter_non_bfrb_classes: train_df = train_df.loc[train_df['is_target']]
if filter_orientation_class_list is not None: train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]
if slice_by_bfrb is True: train_df = train_df.loc[train_df['is_target']]
elif slice_by_bfrb is False: train_df = train_df.loc[~train_df['is_target']]
if slice_by_orientation is not None: train_df = train_df[train_df['orientation'].isin(slice_by_orientation)]

if filter_orientation_class_list is not None or slice_by_orientation is not None:
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()
    train_seqs, test_seqs = train_test_split(sequences['sequence_id'], test_size=(1 - train_size), stratify=sequences[target_col], random_state=random_state)
    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()
else:
    try:
        train_sample_df, hold_out_df = data_utils.sample_balanced_split(train_df, train_pct=train_size, test_pct=min(0.2, 1 - train_size), random_state=random_state)
    except Exception:
        unique_seqs = train_df[["sequence_id", target_col]].drop_duplicates("sequence_id").sample(frac=1, random_state=random_state)
        n_train = int(len(unique_seqs) * train_size)
        train_seqs = unique_seqs.iloc[:n_train]["sequence_id"]
        test_seqs = unique_seqs.iloc[n_train:]["sequence_id"]
        train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
        hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()

X_train, X_test = train_sample_df.copy(), hold_out_df.copy()
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols: cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", "is_target", primary_target]].copy()

groups = X_train["sequence_id"]
print(f"✅ Split Complete: Train={X_train['sequence_id'].nunique()} seqs | Test={X_test['sequence_id'].nunique()} seqs")


✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Train: 2910 seqs | 35.7%
Test:  969 seqs  | 11.9%
✅ Split Complete: Train=2910 seqs | Test=969 seqs


 # 3. Pipeline Initialization

In [4]:
sequence_extractor = SequenceExtractor()

if mode == "single":
    model = V4PrototypicalNetwork(target=single_target)
else:
    model = V4MultiHeadPrototypicalNetwork(primary_target=primary_target, sub_heads=multi_heads)

pipeline = Pipeline([
    (pipe_name, sequence_extractor),
    (proto_name, model),
])


 # 4. EXHAUSTIVE Bayesian Parameter Space

In [5]:
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    param_space = {
        # EXTRACTOR (base_utils_qwen exact match)
        f"{pipe_name}__acc_modes": Categorical(["raw", "smoothed|velocity|jerk", "raw|velocity|displacement|jerk", "smoothed"]),
        f"{pipe_name}__rotation_modes": Categorical(["quaternion", "quaternion|angular_velocity", "quaternion|euler|angular_velocity", "quaternion|delta_euler", "rot6d"]),
        f"{pipe_name}__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled", "raw"]),
        f"{pipe_name}__thm_modes": Categorical(["centered_diff", "diff", "centered", "raw"]),
        f"{pipe_name}__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-3, 10.0, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True, False]),
        f"{pipe_name}__dead_reckoning_detrend": Categorical([True, False]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__window_size": Integer(3, 50),
        f"{pipe_name}__smooth_alpha": Real(0.1, 0.9),
        f"{pipe_name}__clip_value": Categorical([None, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear", "ffill"]),
        f"{pipe_name}__maxlen": Categorical([120, 160, 200, 256]),
        f"{pipe_name}__padding_value": Categorical([-999.0, 0.0]),

        # --- IMU (Accelerometer) Sampling Rates ---
        f"{pipe_name}__imu_native_sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__imu_target_sampling_rate": Categorical([20, 50, 100]),
        
        # --- Rotation (Quaternion/Gyro) Sampling Rates ---
        f"{pipe_name}__rot_native_sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__rot_target_sampling_rate": Categorical([20, 50, 100]),
        
        # --- Time-of-Flight (TOF) Sampling Rates (Typically lower) ---
        f"{pipe_name}__tof_native_sampling_rate": Categorical([5, 10, 20]),
        f"{pipe_name}__tof_target_sampling_rate": Categorical([5, 10, 20]),
        
        # --- Thermopile (THM) Sampling Rates (Typically lower) ---
        f"{pipe_name}__thm_native_sampling_rate": Categorical([5, 10, 20]),
        f"{pipe_name}__thm_target_sampling_rate": Categorical([5, 10, 20]),
        f"{pipe_name}__chunk_window_size": Integer(20, 128),
        f"{pipe_name}__chunk_stride": Integer(10, 64),

        # MODEL BACKBONE & REGULARIZATION
        f"{proto_name}__backbone_type": Categorical(["1dcnn", "2dcnn", "lstm", "attention", "conv4", "resnet12", "resnet18"]),
        f"{proto_name}__filters": Categorical(["32-64", "64-128", "64-128-256", "128-256-256-256", "32-64-128"]),
        f"{proto_name}__kernels": Categorical(["3-3", "5-3", "7-5-3", "5-5-5-5", "3-3-3-3", "9-9-9-9"]),
        f"{proto_name}__pools": Categorical(["none", "2", "2-2", "2-2-2"]),
        f"{proto_name}__lstm_units": Categorical([64, 128, 256]),
        f"{proto_name}__attention_heads": Categorical([2, 4, 8]),
        f"{proto_name}__embed_dim": Categorical([32, 64, 128, 256]),
        f"{proto_name}__dropout": Categorical([0.0]),
        f"{proto_name}__spatial_dropout": Categorical([0.0]),
        f"{proto_name}__l2_reg": Categorical([1e-5]),
        f"{proto_name}__learning_rate": Categorical([5e-4]),
        f"{proto_name}__batch_size": Categorical([32]),
        f"{proto_name}__epochs": Categorical([50]),
        f"{proto_name}__patience": Categorical([10]),
        f"{proto_name}__validation_split": Categorical([0.1]),
        f"{proto_name}__temperature": Real(0.01, 0.5, prior="log-uniform"),
        f"{proto_name}__supcon_weight": Real(0.0, 0.5),
        f"{proto_name}__use_phase_attention": Categorical([True, False]),
        f"{proto_name}__modality_dropout_prob": Categorical([0.0, 0.3, 0.5]),
        f"{proto_name}__use_attention_prototypes": Categorical([True, False]),
        f"{proto_name}__n_way": Categorical([9]),
        f"{proto_name}__n_support": Categorical([5, 40]),
        f"{proto_name}__n_query": Categorical([5, 40]),

        # ALL 11 TEMPORAL AUGMENTATIONS
        f"{proto_name}__use_mixup": Categorical([True, False]),
        f"{proto_name}__mixup_alpha": Real(0.1, 0.8),
        f"{proto_name}__mixup_prob": Real(0.1, 0.9),
        f"{proto_name}__use_time_shift": Categorical([True, False]),
        f"{proto_name}__max_shift_pct": Real(0.05, 0.25),
        f"{proto_name}__use_time_stretch": Categorical([True, False]),
        f"{proto_name}__time_stretch_min": Real(0.7, 0.9),
        f"{proto_name}__time_stretch_max": Real(1.1, 1.3),
        f"{proto_name}__use_gaussian_noise": Categorical([True, False]),
        f"{proto_name}__noise_std": Real(1e-4, 0.05, prior="log-uniform"),
        f"{proto_name}__use_magnitude_scaling": Categorical([True, False]),
        f"{proto_name}__mag_min": Real(0.8, 0.95),
        f"{proto_name}__mag_max": Real(1.05, 1.2),
        f"{proto_name}__use_time_mask": Categorical([True, False]),
        f"{proto_name}__time_mask_ratio": Real(0.05, 0.3),
        f"{proto_name}__use_channel_dropout": Categorical([True, False]),
        f"{proto_name}__channel_drop_prob": Real(0.05, 0.3),
        f"{proto_name}__use_quaternion_flip": Categorical([True, False]),
        f"{proto_name}__quat_flip_prob": Real(0.1, 0.9),
        f"{proto_name}__use_freq_filter": Categorical([True, False]),
        f"{proto_name}__freq_keep_low": Real(0.05, 0.2),
        f"{proto_name}__freq_keep_high": Real(0.8, 0.95),
    }

    if mode == "multi":
        param_space[f"{proto_name}__uncertainty_weighting"] = Categorical([True, False])
        param_space[f"{proto_name}__primary_target"] = Categorical([primary_target])
        param_space[f"{proto_name}__sub_heads"] = Categorical([tuple(multi_heads)])
    else:
        param_space[f"{proto_name}__target"] = Categorical([single_target])

    param_space = prepare_bayesian_space(param_space)

else:
    param_space = {
        f"{pipe_name}__acc_modes": ["raw"],
        f"{pipe_name}__rotation_modes": ["raw"],
        f"{pipe_name}__tof_modes": ["sensor_stats"],
        f"{pipe_name}__thm_modes": ["centered_diff"],
        f"{pipe_name}__motion_filter_mode": [None],

        # --- SAMPLING RATES (Fixed to prevent combinatorial explosion) ---
        f"{pipe_name}__imu_native_sampling_rate": [100],
        f"{pipe_name}__imu_target_sampling_rate": [100],
        f"{pipe_name}__rot_native_sampling_rate": [20],
        f"{pipe_name}__rot_target_sampling_rate": [20],
        f"{pipe_name}__tof_native_sampling_rate": [5],
        f"{pipe_name}__tof_target_sampling_rate": [5],
        f"{pipe_name}__thm_native_sampling_rate": [5],
        f"{pipe_name}__thm_target_sampling_rate": [5],

        f"{pipe_name}__chunk_window_size": [64],
        f"{pipe_name}__chunk_stride": [32],
        f"{pipe_name}__padding_value": [0.0],
        f"{proto_name}__backbone_type": ["conv4", '1dcnn', '2dcnn', 'attention', 'lstm', 'resnet12', 'resnet18'], 
        f"{proto_name}__filters": ["64-128-256"],
        f"{proto_name}__kernels": ["3-3-3"],
        f"{proto_name}__pools": ["2"],
        f"{proto_name}__embed_dim": [128],
        f"{proto_name}__dropout": [0.0],
        f"{proto_name}__spatial_dropout": [0.0],
        f"{proto_name}__l2_reg": [1e-4],
        f"{proto_name}__learning_rate": [5e-4],
        f"{proto_name}__batch_size": [32],
        f"{proto_name}__epochs": [50],
        f"{proto_name}__patience": [15],
        f"{proto_name}__temperature": [0.1],
        f"{proto_name}__supcon_weight": [0.1],
        f"{proto_name}__use_phase_attention": [True],
        f"{proto_name}__modality_dropout_prob": [0.3],
        f"{proto_name}__use_attention_prototypes": [True],
        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [60],
        f"{proto_name}__n_query": [50],
        f"{proto_name}__use_mixup": [False],
        f"{proto_name}__use_time_mask": [False],
        f"{proto_name}__use_gaussian_noise": [False],
    }
    if mode == "single": param_space[f"{proto_name}__target"] = [single_target]
    else:
        param_space[f"{proto_name}__primary_target"] = [primary_target]
        param_space[f"{proto_name}__sub_heads"] = [tuple(multi_heads)]
        param_space[f"{proto_name}__uncertainty_weighting"] = [True]

print(f"✅ Parameter space configured. Total parameters: {len(param_space)}")


✅ Parameter space configured. Total parameters: 40


 # 5. Search Execution

In [ ]:
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        estimator=pipeline, search_spaces=param_space, n_iter=n_iter,
        scoring=competition_scorer, cv=cv_object, n_jobs=n_jobs,
        random_state=random_state, verbose=verbose, refit=True,
        return_train_score=True, error_score=error_score_constant
    )
else:
    search = GridSearchCV(
        estimator=pipeline, param_grid=param_space, scoring=competition_scorer,
        cv=cv_object, n_jobs=n_jobs, verbose=verbose, refit=True,
        return_train_score=True, error_score=error_score_constant,
    )

print(f"🚀 Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\n🏆 Best CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

best_model = search.best_estimator_


🚀 Running grid search...
Fitting 1 folds for each of 7 candidates, totalling 7 fits


 # 6. Save Results & Holdout Evaluation

In [ ]:
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df['mode'] = mode
cv_results_df['search_mode'] = search_mode
cv_results_df['timestamp'] = timestamp
results_path = results_dir / f"cv_results_{mode}_{search_mode}_{timestamp}.csv"
cv_results_df.to_csv(results_path, index=False)
print(f"✅ CV results saved to: {results_path}")

y_test_true = hold_out_df[["sequence_id", "is_target", target_col]].copy()
y_pred = best_model.predict(X_test)

holdout_eval = evaluate_holdout(y_test_true, y_pred, target_col=primary_target if mode == "multi" else target_col)
print(f"\n🎯 Holdout Competition Score: {holdout_eval['competition_score']:.4f}")

holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_eval["results_df"].to_csv(holdout_results_path, index=False)
print(f"✅ Holdout predictions saved to: {holdout_results_path}")

In [ ]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit

from sklearn.metrics import f1_score, classification_report

import data_utils
from base_utils_qwen import (
    SequenceExtractor,
    prepare_multitask_param_space,
    prepare_bayesian_space,
    competition_scorer as bfrb_competition_scorer,
    evaluate_holdout,
    plot_training_curves,
)
from proto_utils_qwen import SingleHeadPrototypicalNetwork, MultiHeadPrototypicalNetwork
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, make_scorer

In [ ]:
# ============================================================
# CONFIGURATION - CHANGE THIS TO SWITCH MODES
# ============================================================

# Set this to "single" or "multi"
mode = "single"  # "single" for binary+gesture, "multi" for multi-head

single_target = "bfrb"  # For single mode: "gesture", "gesture_action", etc.
target_col = single_target

# For MULTI mode: which heads to predict
multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "bfrb"  # For multi mode: what to evaluate F1 on

pipe_name = "extractor"
proto_name = "model"

search_mode = "bayesian"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
n_jobs = 1
train_size = 0.4
error_score_constant = 0.0
verbose = 1
do_cross_val = False

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)  # ✅ Fast

if target_col == 'bfrb':
    competition_scorer = bfrb_competition_scorer
else:
    competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

filter_non_brb_classes = False
filter_orientation_class_list = None

print(f"Mode: {mode}")
print(f"Search mode: {search_mode}")

In [ ]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

In [ ]:
train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)
q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
bad_q = norm.squeeze() == 0
q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
q_wxyz = q_wxyz / norm

q_xyzw = q_wxyz[:, [1, 2, 3, 0]]
euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)
euler_xyz[:, [1, 2]] *= -1.0
q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()
q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]
train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed

print(f"left-handed corrected: {left_handed_mask.sum()} rows")

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
print(f"upside-down corrected: {upside_down_mask.sum()} rows")

train_df = train_df.drop(columns=["handedness"])

In [ ]:
# Create target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

if filter_non_brb_classes:
    train_df = train_df.loc[train_df['is_target'],:]

if filter_orientation_class_list is not None:
    train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]

    # Get unique sequences
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()

    # Stratified split by your target column
    train_seqs, test_seqs = train_test_split(
        sequences['sequence_id'], 
        test_size=(1 - train_size), 
        stratify=sequences[target_col],  # or use multiple columns
        random_state=random_state
    )

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]
else:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
    )


X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

In [ ]:
train_sample_df.groupby([target_col, 'orientation']).agg(
    {'sequence_id':'nunique'})

In [ ]:
sequence_extractor = SequenceExtractor(
    acc_modes='raw'
)

if mode == "single":
    # BinaryPlusGesturePrototypicalNetwork uses is_target and gesture automatically
    model = SingleHeadPrototypicalNetwork(
        target=single_target,
    )
else:
    model = MultiHeadPrototypicalNetwork(
        primary_target=primary_target,
    )

pipeline = Pipeline([
    ("extractor", sequence_extractor),
    ("model", model),
])

In [ ]:
# ============================================================================
# PARAMETER SPACE DEFINITION
# ============================================================================
if search_mode == "bayesian":
    base_extractor_params = {
        f"{pipe_name}__acc_modes": Categorical(["smoothed|velocity|jerk", "raw"]),
        f"{pipe_name}__rotation_modes": Categorical(["quaternion|euler|angular_velocity", "quaternion|delta_euler"]),
        f"{pipe_name}__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"]),
        f"{pipe_name}__thm_modes": Categorical(["centered_diff", "diff"]),
        f"{pipe_name}__motion_filter_mode": Categorical(["kalman", "extended_kalman", None]),
        f"{pipe_name}__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-2, 1e-1, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True, False]),
        f"{pipe_name}__dead_reckoning_detrend": Categorical([True, False]),
        # CRITICAL: Rank 1 used 20Hz! Lower sampling acts as a powerful regularizer.
        f"{pipe_name}__sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0, 100.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear", "ffill"]),
        f"{pipe_name}__window_size": Integer(10, 50),
        f"{pipe_name}__smooth_alpha": Real(0.1, 0.9),
        f"{pipe_name}__maxlen": Categorical([120, 160, 200]),
        f"{pipe_name}__padding_value": Categorical([-999.0]),
    }

    base_model_params = {
        f"{proto_name}__backbone_type": Categorical(["1dcnn"]),
        f"{proto_name}__filters": Categorical(["64-128-256", "32-64-128", "64-128"]),
        f"{proto_name}__kernels": Categorical(["5-3", "7-5-3", "3-3"]),
        f"{proto_name}__pools": Categorical(["none", "2", "2-2"]),

        # Regularization narrowed around the ~0.45 sweet spot
        f"{proto_name}__dropout": Real(0.35, 0.55),
        f"{proto_name}__spatial_dropout": Real(0.1, 0.3),  # NEW
        f"{proto_name}__l2_reg": Real(1e-5, 1e-3, prior="log-uniform"),  # NEW
        f"{proto_name}__temperature": Real(0.05, 0.2, prior="log-uniform"),  # NEW
        f"{proto_name}__embed_dim": Categorical([64, 128]),

        # Learning rate narrowed around the 3.5e-3 winner
        f"{proto_name}__learning_rate": Real(1e-3, 5e-3, prior="log-uniform"),
        f"{proto_name}__batch_size": Categorical([16, 32]),
        f"{proto_name}__epochs": Categorical([50]),
        f"{proto_name}__patience": Categorical([10, 15]),

        f"{proto_name}__n_way": Categorical([9]),
        f"{proto_name}__n_support": Categorical([20, 40, 60]),
        f"{proto_name}__n_query": Categorical([20, 40, 60]),

        f"{proto_name}__use_mixup": Categorical([True, False]),
        f"{proto_name}__mixup_alpha": Real(0.2, 0.5),
        f"{proto_name}__use_time_mask": Categorical([True]),
        f"{proto_name}__time_mask_ratio": Real(0.10, 0.20),
        f"{proto_name}__use_gaussian_noise": Categorical([True]),
        f"{proto_name}__noise_std": Real(0.001, 0.01, prior="log-uniform"),
    }

    if mode == "single":
        param_space = {**base_extractor_params, **base_model_params, f"{proto_name}__target": Categorical([single_target])}
    else:
        heads_tuple = tuple(multi_heads)
        param_space = {**base_extractor_params, **base_model_params, f"{proto_name}__sub_heads": Categorical([heads_tuple]), f"{proto_name}__primary_target": Categorical([primary_target])}

    param_space = prepare_bayesian_space(param_space)

else:  # Grid Search (Tightly constrained to proven winners)
    param_space = {
        f"{pipe_name}__acc_modes": ["smoothed|velocity|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|euler|angular_velocity"],
        f"{pipe_name}__tof_modes": ["sensor_stats"],
        f"{pipe_name}__thm_modes": ["centered_diff"],
        f"{pipe_name}__motion_filter_mode": ["kalman"],
        f"{pipe_name}__sampling_rate": [20, 100],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__window_size": [20, 40],
        f"{pipe_name}__maxlen": [160],
        f"{pipe_name}__padding_value": [-999.0],

        f"{proto_name}__backbone_type": ["1dcnn"],
        f"{proto_name}__filters": ["64-128-256", "32-64-128"],
        f"{proto_name}__kernels": ["5-3"],
        f"{proto_name}__pools": ["2", "none"],

        f"{proto_name}__dropout": [0.45, 0.50],
        f"{proto_name}__spatial_dropout": [0.15, 0.20],  # NEW
        f"{proto_name}__l2_reg": [1e-4],  # NEW
        f"{proto_name}__temperature": [0.1],  # NEW
        f"{proto_name}__embed_dim": [64, 128],
        f"{proto_name}__lstm_units": [128],

        f"{proto_name}__learning_rate": [1e-3, 3e-3, 5e-3],
        f"{proto_name}__batch_size": [16, 32],
        f"{proto_name}__epochs": [50],
        f"{proto_name}__patience": [15],

        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [40],
        f"{proto_name}__n_query": [40],

        f"{proto_name}__use_mixup": [False],
        f"{proto_name}__use_time_mask": [True],
        f"{proto_name}__time_mask_ratio": [0.15],
        f"{proto_name}__use_gaussian_noise": [True],
        f"{proto_name}__noise_std": [0.005, 0.01],
    }

    if mode == "single":
        param_space[f"{proto_name}__target"] = [single_target]
    else:
        param_space[f"{proto_name}__primary_target"] = [primary_target]
        param_space[f"{proto_name}__sub_heads"] = [tuple(multi_heads)]


In [ ]:
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    # For multi mode, include all heads plus the primary target for lookup
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols:
        cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", primary_target]].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
groups = X_train["sequence_id"]

print(f"MODE: {mode}")
print(f"Single target: {single_target if mode=='single' else 'N/A'}")
print(f"Multi heads: {multi_heads if mode=='multi' else 'N/A'}")
print(f"Primary target: {primary_target if mode=='multi' else 'N/A'}")

print(f"y_train columns: {y_train.columns.tolist()}")

In [ ]:
if search_mode == "bayesian":

    param_space = prepare_multitask_param_space(param_space, search_mode)
    
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=competition_scorer,
        cv=cv_object,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=competition_scorer,
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

In [ ]:
# ============================================================
# SAVE CV RESULTS TO CSV
# ============================================================

# Convert search results to DataFrame
cv_results_df = pd.DataFrame(search.cv_results_)

# Add metadata columns
cv_results_df['mode'] = mode
cv_results_df['search_mode'] = search_mode
cv_results_df['timestamp'] = timestamp

# Define save path
results_path = results_dir / f"cv_results_{mode}_{search_mode}_{timestamp}.csv"

# Save to CSV
cv_results_df.to_csv(results_path, index=False)

print(f"CV results saved to: {results_path}")
print(f"Shape: {cv_results_df.shape}")
print(f"Columns: {cv_results_df.columns.tolist()}")

In [ ]:
# ============================================================
# EVALUATE ON HOLDOUT TEST SET - WORKS FOR BOTH MODES
# ============================================================
y_test_true = hold_out_df[["sequence_id", "is_target", target_col]].copy()
y_pred = best_model.predict(X_test)

holdout_eval = evaluate_holdout(y_test_true, y_pred, target_col=primary_target)
competition_score = holdout_eval["competition_score"]

holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_eval["results_df"].to_csv(holdout_results_path, index=False)
print(f"\nHoldout predictions saved to: {holdout_results_path}")

In [ ]:
# ============================================================
# TRAINING / VALIDATION LOSS & ACCURACY CURVES
# ============================================================
proto_clf = best_model.named_steps[proto_name]
if hasattr(proto_clf, "history_") and proto_clf.history_:
    plot_training_curves(
        proto_clf.history_,
        title=f"Prototypical Network ({mode} mode)",
    )
    plt.show()
else:
    print("No training history found on best model — history_ is populated after .fit()")

# Contrastive Siamese Network for Sensor-Based Gesture Recognition

This notebook implements a **Supervised Contrastive Siamese Network** for the Kaggle challenge. 
It utilizes the temporal `SequenceExtractor` from `base_utils_qwen` to generate rich multi-domain features.

**Architecture Options:**
- **Backbone:** 1D CNN with dilated/standard convolutions.
- **Temporal Aggregation:** Bidirectional GRU (`gru`), Transformer Encoder (`attention`), or Global Pooling (`pool`).
- **Sequence Masking:** Automatically masks padded timesteps so temporal models don't process noise.

We provide both `GridSearchCV` and `BayesSearchCV` pipelines.

In [ ]:
import sys
import os
import warnings
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
import random
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, train_test_split

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

try:
    dataset_name = os.listdir("/kaggle/input/datasets/keithmarange")[0]
    sys.path.append(f"/kaggle/input/datasets/keithmarange/{dataset_name}/")
    sys.path.append("/kaggle/input/cmi-competition-code")
except Exception:
    pass

# ============================================================
# SETUP PATHS – MUST BE FIRST CELL
# ============================================================

import sys
import os

# Figure out where we are
current_dir = os.getcwd()
workspace_root = current_dir

# If we're in notebooks/ subfolder, go up one level
if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)

# Add paths so Python can find src/
src_path = os.path.join(workspace_root, "src")
sys.path.insert(0, workspace_root)   # So 'import src' works
sys.path.insert(0, src_path)         # So 'import data_utils' works directly

# ============================================================
# IMPORTS
# ============================================================

# Try importing with src/ prefix (local)
try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    from src.utils_siamese_contrastive import KerasContrastiveSiameseClassifier
    print("✅ Imports loaded from src/")
    
except ImportError:
    # Fallback: flattened imports (Kaggle or if src not found)
    try:
        import data_utils
        from base_utils_qwen import (
            SequenceExtractor,
            competition_scorer,
            evaluate_holdout,
            make_competition_scorer,
            prepare_bayesian_space,
        )
        from utils_siamese_contrastive import KerasContrastiveSiameseClassifier
        print("✅ Imports loaded flattened")
    except ImportError as e:
        print(f"❌ Import failed: {e}")
        print(f"Current directory: {os.getcwd()}")
        print(f"Files in current directory: {os.listdir('.')}")
        print(f"Files in src/ (if exists): {os.listdir('src') if os.path.exists('src') else 'src/ not found'}")
        raise

from utils_siamese_contrastive import KerasContrastiveSiameseClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer
from skopt.space import Categorical, Integer, Real
from base_utils_qwen import prepare_bayesian_space
import random
from datetime import datetime

✅ Imports loaded from src/


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "bayesian"  # "grid" or "bayesian"

random_state = 42
n_splits = 3
n_iter = 20
train_size = 0.4
error_score_constant = 0.0
verbose = 3
do_cross_val = False

# Fast local smoke test on data/eg.csv (set False for full train.csv)
use_eg_sample = False
eg_sample_pct = 0.02

# Optional row-level filters before split
do_handness = False
filter_non_bfrb_classes = False
filter_orientation_class_list = None  # e.g. ["Seated Straight"]
slice_by_orientation = None
slice_by_bfrb = None

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if target_col == "bfrb":
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

# Smaller training budget when using the eg.csv sample
default_epochs = 8 if use_eg_sample else 30
default_patience = 3 if use_eg_sample else 8
default_batch_size = 16 if use_eg_sample else 32

print(f"Search mode: {search_mode}")
print(f"use_eg_sample: {use_eg_sample}")

In [ ]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    print(f"Using eg.csv sample: {raw_train_df['sequence_id'].nunique()} sequences")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(
            frac=eg_sample_pct, random_state=random_state
        )
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()
        print(f"Using {eg_sample_pct:.0%} sequence sample: {raw_train_df['sequence_id'].nunique()} sequences")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness and "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    train_df = train_df.drop(columns=["handedness"])

upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

if filter_non_bfrb_classes:
    train_df = train_df.loc[train_df["is_target"]].copy()

if filter_orientation_class_list is not None:
    train_df = train_df[train_df[orientation_col].isin(filter_orientation_class_list)].copy()

if slice_by_bfrb is True:
    train_df = train_df.loc[train_df["is_target"]].copy()
elif slice_by_bfrb is False:
    train_df = train_df.loc[~train_df["is_target"]].copy()

if slice_by_orientation is not None:
    train_df = train_df[train_df[orientation_col].isin(slice_by_orientation)].copy()

if filter_orientation_class_list is not None:
    sequences = train_df[["sequence_id", "is_target", target_col, orientation_col]].drop_duplicates()
    train_seqs, test_seqs = train_test_split(
        sequences["sequence_id"],
        test_size=(1 - train_size),
        stratify=sequences[target_col],
        random_state=random_state,
    )
    train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()
else:
    try:
        train_sample_df, hold_out_df = data_utils.sample_balanced_split(
            train_df,
            train_pct=train_size,
            test_pct=min(0.2, 1 - train_size),
            random_state=random_state,
        )
    except ValueError as exc:
        print(f"Balanced split unavailable ({exc}); using simple sequence split.")
        unique_seqs = (
            train_df[["sequence_id", target_col]]
            .drop_duplicates("sequence_id")
            .sample(frac=1, random_state=random_state)
        )
        n_train = max(1, int(len(unique_seqs) * train_size))
        train_seqs = unique_seqs.iloc[:n_train]["sequence_id"]
        test_seqs = unique_seqs.iloc[n_train:]["sequence_id"]
        if len(test_seqs) == 0 and len(unique_seqs) > 1:
            test_seqs = unique_seqs.iloc[-1:]["sequence_id"]
            train_seqs = unique_seqs.iloc[:-1]["sequence_id"]
        train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
        hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()
        print(
            f"Train: {train_sample_df['sequence_id'].nunique()} seqs | "
            f"Test: {hold_out_df['sequence_id'].nunique()} seqs"
        )

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique()}")
print(f"X_train rows: {len(X_train):,} | y_train rows: {len(y_train):,}")

In [ ]:
# ============================================================
# GRID SEARCH PARAMETER SPACE (One Example Each)
# ============================================================

extractor_space_grid = {
    "extractor__acc_modes": ["raw|velocity|jerk"],
    "extractor__rotation_modes": ["quaternion"],
    "extractor__tof_modes": ["sensor_stats"],
    "extractor__thm_modes": ["centered_diff"],
    "extractor__window_size": [5],
    "extractor__clip_value": [None],
    "extractor__interp_mode": ["linear"],
    "extractor__motion_filter_mode": [None],
    "extractor__kalman_process_noise": [1e-3],
    "extractor__kalman_measurement_noise": [1e-1],
    "extractor__padding_value": [0.0],
    # NEW parameters (add one example each)
    "extractor__native_sampling_rate": [10, 100],
    "extractor__target_sampling_rate": [20, 100, 200],
    "extractor__chunk_window_size": [128],
    "extractor__chunk_stride": [64],
    "extractor__use_dead_reckoning": [False],
    "extractor__dead_reckoning_detrend": [False],
}

classifier_space_grid = {
    "classifier__target": ["bfrb"],
    "classifier__maxlen": [160],
    "classifier__padding_value": [0.0],
    "classifier__backbone_filters": ["64-128"],
    "classifier__kernel_sizes": ["3-3"],
    "classifier__temporal_mode": ["gru"],
    "classifier__embedding_dim": [64],
    "classifier__contrastive_weight": [0.5],
    "classifier__temperature": [0.1],
    "classifier__dense_units": ["64"],
    "classifier__dropout": [0.3],
    "classifier__learning_rate": [1e-3],
    "classifier__batch_size": [32],
    "classifier__epochs": [50],
    "classifier__patience": [10],
    "classifier__verbose": [0],
    "classifier__random_state": [42],
}

# ============================================================
# BAYESIAN PARAMETER SPACE (Full Exploration)
# ============================================================

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    param_space = {
        # ---------- EXTRACTOR ----------
        # Sensor modes (keep fixed to best found, but can change if needed)
        "extractor__acc_modes": Categorical(['smoothed|velocity|jerk']),
        "extractor__rotation_modes": Categorical(['quaternion|angular_velocity']),
        "extractor__tof_modes": Categorical(['sensor_stats']),
        "extractor__thm_modes": Categorical(['centered']),
        # Preprocessing
        "extractor__window_size": Integer(3, 25, prior="uniform"),
        "extractor__clip_value": Categorical([None]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman", 'extended_kalman']),
        "extractor__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-3, 10.0, prior="log-uniform"),
        "extractor__padding_value": Categorical([0.0]),
        # NEW parameters
        "extractor__native_sampling_rate": Categorical([10, 100]),  # usually fixed to raw rate
        "extractor__target_sampling_rate": Categorical([20, 50, 100, 200]),
        "extractor__chunk_window_size": Integer(64, 256, prior="uniform"),
        "extractor__chunk_stride": Integer(32, 128, prior="uniform"),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        
        # ---------- CLASSIFIER ----------
        "classifier__target": Categorical([target_col]),        # should match extractor maxlen
        "classifier__padding_value": Categorical([0.0]),
        "classifier__backbone_filters": Categorical(["32-64", "64-128", "32-64-128", "64-128-256"]),
        "classifier__kernel_sizes": Categorical(["3-3", "5-3", "3-5-3"]),
        "classifier__temporal_mode": Categorical(['attention', 'pool','gru']),
        "classifier__embedding_dim": Integer(32, 512, prior="uniform"),
        "classifier__contrastive_weight": Real(0.05, 0.95, prior="uniform"),
        "classifier__temperature": Real(0.01, 0.5, prior="log-uniform"),
        "classifier__dense_units": Categorical(["32", "64", "32-64", "64-32"]),
        "classifier__dropout": Real(0.05, 0.4, prior="uniform"),
        "classifier__learning_rate": Categorical([3.7e-3]),   # fixed to best found
        "classifier__batch_size": Categorical([16, 32, 64, 128]),
        "classifier__epochs": Categorical([50]),
        "classifier__patience": Categorical([10]),
        "classifier__verbose": Categorical([3]),
        "classifier__random_state": Categorical([42]),
    }
    
    # Convert complex objects to JSON strings for skopt
    param_space = prepare_bayesian_space(param_space)
    
else:
    # GRID SEARCH MODE - One example each
    param_space = {**extractor_space_grid, **classifier_space_grid}

In [ ]:

# ============================================================
# PRINT SUMMARY
# ============================================================

print("=" * 60)
print("PARAMETER SPACE SUMMARY - SIAMESE NETWORK")
print("=" * 60)
print(f"Search Mode: {search_mode}")
print(f"Target Column: {target_col}")

if search_mode == "bayesian":
    print("Bayesian mode: Continuous parameter sampling")
    print(f"Total extractor parameters: {len(extractor_space_grid)}")
    print(f"Total classifier parameters: {len(classifier_space_grid)}")
    print("\n--- Feature Extractor Options (Bayesian) ---")
    print(f"  ACC_MODES: fixed to 'smoothed|velocity|jerk'")
    print(f"  ROTATION_MODES: fixed to 'quaternion|angular_velocity'")
    print(f"  TOF_MODES: fixed to 'sensor_stats'")
    print(f"  THM_MODES: fixed to 'centered'")
    print(f"  Window size: {param_space['extractor__window_size']}")
    print(f"  Motion filter: {param_space['extractor__motion_filter_mode']}")
    print(f"  Target sampling rate: {param_space['extractor__target_sampling_rate']}")
    print(f"  Chunk window size: {param_space['extractor__chunk_window_size']}")
    print(f"  Chunk stride: {param_space['extractor__chunk_stride']}")
    print(f"  Use dead reckoning: {param_space['extractor__use_dead_reckoning']}")
    print("\n--- Siamese Network Options (Bayesian) ---")
    print(f"  backbone_filters: {param_space['classifier__backbone_filters']}")
    print(f"  kernel_sizes: {param_space['classifier__kernel_sizes']}")
    print(f"  temporal_mode: {param_space['classifier__temporal_mode']}")
    print(f"  embedding_dim: {param_space['classifier__embedding_dim']}")
    print(f"  contrastive_weight: {param_space['classifier__contrastive_weight']}")
    print(f"  temperature: {param_space['classifier__temperature']}")
    print(f"  dropout: {param_space['classifier__dropout']}")
    print(f"  learning_rate: {param_space['classifier__learning_rate']} (fixed)")
    print(f"  batch_size: {param_space['classifier__batch_size']}")
else:
    print("Grid mode: Fixed single configuration")
    print(f"Total extractor parameters: {len(extractor_space_grid)}")
    print(f"Total classifier parameters: {len(classifier_space_grid)}")
    total_combos = 1
    for key, values in param_space.items():
        total_combos *= len(values)
    print(f"Total grid combinations: {total_combos:,}")
    print("\n--- Grid Parameters (One Example Each) ---")
    for key, value in param_space.items():
        print(f"  {key}: {value}")

print("\n✅ Parameter space ready for optimization.")

In [ ]:
# ============================================================
# EXTRACTOR CHANNEL PREVIEW
# ============================================================
sample_df = X_train.head(min(2000, len(X_train)))
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]
is_bayesian = search_mode == "bayesian" and SKOPT_AVAILABLE

if is_bayesian:

    print("Bayesian mode: sampling 3 extractor configs for channel preview\n")
    preview_combos = []
    for _ in range(3):
        params = {}
        for key in extractor_keys:
            param_name = key.replace("extractor__", "")
            space = param_space[key]
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                value = random.uniform(space.low, space.high)
            else:
                value = space
            params[param_name] = value
        params["padding_value"] = 0.0
        preview_combos.append(params)
else:
    extractor_values = [param_space[k] for k in extractor_keys]
    preview_combos = [
        {k.replace("extractor__", ""): v for k, v in zip(extractor_keys, combo)}
        for combo in itertools.islice(itertools.product(*extractor_values), 3)
    ]
    for params in preview_combos:
        params["padding_value"] = 0.0

for i, extractor_params in enumerate(preview_combos, start=1):
    try:
        extractor = SequenceExtractor(**extractor_params)
        extractor.fit(sample_df)
        out = extractor.transform(sample_df)
        n_channels = out["X"].shape[2]
        key_params = {k: extractor_params[k] for k in ["acc_modes", "rotation_modes", "maxlen", "sampling_rate"] if k in extractor_params}
        print(f"Preview {i}: {key_params} -> channels={n_channels}, timesteps={out['X'].shape[1]}")
    except Exception as exc:
        print(f"Preview {i} failed: {exc}")

In [ ]:
# ============================================================
# PIPELINE & SEARCH
# ============================================================
pipe = Pipeline([
    ("extractor", SequenceExtractor(padding_value=0.0)),
    (
        "classifier",
        KerasContrastiveSiameseClassifier(
            target=target_col,
            epochs=default_epochs,
            patience=default_patience,
            batch_size=default_batch_size,
            verbose=0,
            random_state=random_state,
        ),
    ),
])

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        pipe,
        param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        error_score=error_score_constant,
        verbose=verbose,
        return_train_score=True,
    )
else:
    search = GridSearchCV(
        pipe,
        param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        error_score=error_score_constant,
        verbose=verbose,
        return_train_score=True,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

In [ ]:
# Create a timestamp for the filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- 1. Save the full cv_results_ DataFrame ---
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df.to_csv(f"siamese_cv_results_{timestamp}.csv", index=False)
print(f"✅ Full CV results saved to siamese_cv_results_{timestamp}.csv")

# --- 2. Save a summary of the best parameters and scores ---
best_summary = {
    "best_score": search.best_score_,
    "best_params": str(search.best_params_),
    "mean_fit_time": search.cv_results_['mean_fit_time'][search.best_index_],
    "std_fit_time": search.cv_results_['std_fit_time'][search.best_index_],
    "mean_test_score": search.cv_results_['mean_test_score'][search.best_index_],
    "std_test_score": search.cv_results_['std_test_score'][search.best_index_],
}
best_df = pd.DataFrame([best_summary])
best_df.to_csv(f"siamese_best_summary_{timestamp}.csv", index=False)
print(f"✅ Best summary saved to siamese_best_summary_{timestamp}.csv")

# --- 3. (Optional) Save the best parameters as a readable text file ---
with open(f"siamese_best_params_{timestamp}.txt", "w") as f:
    f.write(f"Best CV Score: {search.best_score_:.4f}\n")
    f.write("Best Parameters:\n")
    for key, value in search.best_params_.items():
        f.write(f"  {key}: {value}\n")
print(f"✅ Best parameters saved to siamese_best_params_{timestamp}.txt")

print("\n✅ All CV results saved successfully.")

In [ ]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================
if not do_cross_val and len(X_test) > 0:
    y_pred = best_model.predict(X_test)
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")

    if target_col == "bfrb":
        eval_dict = evaluate_holdout(y_test_seq, y_pred, target_col=target_col, verbose=True)
        print(f"Holdout Competition Score: {eval_dict['competition_score']:.4f}")
    else:
        print("\nClassification Report:")
        print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))
        macro_f1 = f1_score(y_test_seq[target_col], y_pred, average="macro", zero_division=0)
        print(f"Macro F1 Score: {macro_f1:.4f}")
else:
    print("Holdout evaluation skipped (cross-val mode or empty test set).")

In [ ]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
# %pip install xgboost sktime deep-forest scikit-optimize

# Suppress verbose logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

# Attempt to import deep-forest (install via: pip install deep-forest)
try:
    from deepforest import CascadeForestClassifier
    DEEP_FOREST_AVAILABLE = True
except ImportError:
    DEEP_FOREST_AVAILABLE = False
    print("Warning: deep-forest not installed. Install via 'pip install deep-forest'")

# Custom utilities
import data_utils
from v1_vibration_extractor import V1CargoExtractor

from base_utils_qwen import competition_scorer as bfrb_competition_scorer, evaluate_holdout, plot_training_curves
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, make_scorer

import xgboost as xgb

# Add this to your existing imports in Cell 1
try:
    from sktime.transformations.panel.rocket import MiniRocket
    from sklearn.linear_model import RidgeClassifier
    ROCKET_AVAILABLE = True
except ImportError:
    ROCKET_AVAILABLE = False
    print("Warning: sktime not installed. Install via 'pip install sktime'")

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from baselines_utils import (
    ManyToOneWrapper, 
    ManyToOneWrapperTemporal, 
    make_baseline_pipeline, 
    build_feature_extractor
)

In [ ]:
# ============================================================
# CONFIGURATION — switch feature + classifier pipelines here
# ============================================================
TRAIN_DUMMY = False
TRAIN_RF = True
TRAIN_XGB = True
TRAIN_DEEP_FOREST = True
# Add this alongside TRAIN_RF, TRAIN_XGB, etc.
TRAIN_ROCKET = True

# Feature extraction mode
FEATURE_MODE = "cargo_spectral" 

# Tabular augmentation (optional)
use_tabular_augment = False

target_col = "orientation"  # or 'bfrb' depending on your target
search_mode = "bayes"    # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 15
train_size = 0.5       # lower for quick local tests; raise for full runs
error_score_constant = 0.0
verbose = 2
do_cross_val = False

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)

if target_col == 'bfrb':
    competition_scorer = bfrb_competition_scorer
else:
    competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

filter_non_brb_classes = True  
filter_orientation_class_list = None

In [ ]:
# Install all required packages
!pip install -q xgboost scikit-optimize sktime deep-forest

# Verify installations
import xgboost
import skopt
print(f"XGBoost version: {xgboost.__version__}")
print(f"scikit-optimize version: {skopt.__version__}")

In [ ]:
# ============================================================
# DATA LOADING & PREPROCESSING
# ============================================================
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
train_df = train_df.drop(columns=["handedness"])

# Targets
train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]


if filter_non_brb_classes:
    train_df = train_df.loc[train_df['is_target'],:]

if filter_orientation_class_list is not None:
    train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]

    # Get unique sequences
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()

    # Stratified split by your target column
    train_seqs, test_seqs = train_test_split(
        sequences['sequence_id'], 
        test_size=(1 - train_size), 
        stratify=sequences[target_col],  # or use multiple columns
        random_state=random_state
    )

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]
else:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df, train_pct=train_size, test_pct=min(0.18, 1 - train_size), random_state=random_state
    )


X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = train_sample_df[["sequence_id", "is_target", target_col]].copy()
y_test = hold_out_df[["sequence_id", "is_target", target_col]].copy()
groups = X_train["sequence_id"]

In [ ]:
# ============================================================================
# PARAMETER SPACE DEFINITION — INCLUDES FEATURE EXTRACTION & CLASSIFIERS
# ============================================================================

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    
    # ===== SHARED TABULAR EXTRACTOR (Random Forest, XGBoost, Deep Forest) =====
    tabular_extractor_bayes_space = {
        "extractor__sampling_rate": Integer(20, 200),
        "extractor__acc_modes": Categorical(["raw", "raw|velocity", "smoothed|velocity|jerk", 'jerk']),
        "extractor__rotation_modes": Categorical(["quaternion", "quaternion|angular_velocity", "quaternion|euler"]),
        "extractor__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled_diff"]),
        "extractor__thm_modes": Categorical(["centered_diff", "diff", "centered", "raw"]),
        "extractor__window_size": Integer(10, 200),
        "extractor__clip_value": Categorical([None, 50.0, 100.0, 150.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
    }

    # ===== SHARED TEMPORAL EXTRACTOR (MiniRocket) =====
    temporal_extractor_bayes_space = {
        "extractor__sampling_rate": Integer(20, 200),
        "extractor__acc_modes": Categorical(["raw", "raw|velocity", "smoothed|velocity|jerk", 'jerk']),
        "extractor__rotation_modes": Categorical(["quaternion", "quaternion|angular_velocity"]),
        "extractor__tof_modes": Categorical(["pooled_stats", "sensor_stats"]),
        "extractor__thm_modes": Categorical(["centered_diff", "raw"]),
        "extractor__maxlen": Integer(30, 300),
        "extractor__window_size": Integer(10, 50),
        "extractor__clip_value": Categorical([None, 50.0, 100.0, 150.0]),
        "extractor__interp_mode": Categorical(["linear", "ffill"]),
        "extractor__motion_filter_mode": Categorical([None, "kalman"]),
        "extractor__use_dead_reckoning": Categorical([True, False]),
        "extractor__dead_reckoning_detrend": Categorical([True, False]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
    }

    # ===== 1. Random Forest =====
    rf_param_space = {
        **tabular_extractor_bayes_space,
        "classifier__base_estimator__n_estimators": Integer(10, 5000),
        "classifier__base_estimator__max_depth": Integer(5, 200),
        "classifier__base_estimator__min_samples_split": Integer(2, 200),
        "classifier__base_estimator__min_samples_leaf": Integer(1, 200),
        "classifier__base_estimator__max_features": Categorical(["sqrt", "log2", None]),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }

    # ===== 2. XGBoost =====
    xgb_param_space = {
        **tabular_extractor_bayes_space,
        "classifier__base_estimator__n_estimators": Integer(5, 2000),
        "classifier__base_estimator__learning_rate": Real(1e-3, 9e-1, prior="log-uniform"),
        "classifier__base_estimator__max_depth": Integer(3, 1000),
        "classifier__base_estimator__subsample": Real(0.6, 1.0, prior="uniform"),
        "classifier__base_estimator__colsample_bytree": Real(0.6, 1.0, prior="uniform"),
        "classifier__base_estimator__min_child_weight": Integer(1, 50),
    }

    # ===== 3. Deep Forest =====
    df_param_space = {
        **tabular_extractor_bayes_space,
        "classifier__base_estimator__n_estimators": Integer(2, 5000),       # Number of cascades
        "classifier__base_estimator__max_depth": Integer(5, 5000),          # Max depth of trees
        "classifier__base_estimator__n_trees": Integer(50, 5000),          # Number of trees per layer
        "classifier__base_estimator__criterion": Categorical(["gini", "entropy"]),
    }

    # ===== 4. MiniRocket =====
    rocket_param_space = {
        **temporal_extractor_bayes_space,
        "classifier__base_estimator__num_kernels": Integer(84, 15000),
        "classifier__base_estimator__alpha": Real(1e-4, 1e3, prior="log-uniform"),
        "classifier__base_estimator__feature_selection_percentile": Categorical([None, 25, 50, 75, 90]),
        "classifier__base_estimator__class_weight": Categorical(["balanced", None]),
    }

else:  # GRID SEARCH
    
    # ===== SHARED TABULAR EXTRACTOR (Random Forest, XGBoost, Deep Forest) =====
    tabular_extractor_grid_space = {
        "extractor__sampling_rate": [50],
        "extractor__acc_modes": ["smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__window_size": [20],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
    }

    # ===== SHARED TEMPORAL EXTRACTOR (MiniRocket) =====
    temporal_extractor_grid_space = {
        "extractor__sampling_rate": [50],
        "extractor__acc_modes": ["smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity"],
        "extractor__tof_modes": ["pooled_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__maxlen": [150],
        "extractor__window_size": [40],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__use_dead_reckoning": [False],
        "extractor__dead_reckoning_detrend": [False],
        "extractor__kalman_process_noise": [1e-3],
        "extractor__kalman_measurement_noise": [1e-1],
    }

    # ===== 1. Random Forest =====
    rf_param_space = {
        **tabular_extractor_grid_space,
        "classifier__base_estimator__n_estimators": [10],
        "classifier__base_estimator__max_depth": [4],
        "classifier__base_estimator__min_samples_split": [3],
        "classifier__base_estimator__min_samples_leaf": [5],
        "classifier__base_estimator__max_features": ["sqrt"],
        "classifier__base_estimator__class_weight": ["balanced"],
    }

    # ===== 2. XGBoost =====
    xgb_param_space = {
        **tabular_extractor_grid_space,
        "classifier__base_estimator__n_estimators": [20],
        "classifier__base_estimator__learning_rate": [0.05],
        "classifier__base_estimator__max_depth": [4],
        "classifier__base_estimator__subsample": [0.8],
        "classifier__base_estimator__colsample_bytree": [0.8],
        "classifier__base_estimator__min_child_weight": [1],
    }

    # ===== 3. Deep Forest =====
    df_param_space = {
        **tabular_extractor_grid_space,
        "classifier__base_estimator__n_estimators": [3],
        "classifier__base_estimator__max_depth": [10],
        "classifier__base_estimator__n_trees": [10],
        "classifier__base_estimator__criterion": ["gini"],
    }

    # ===== 4. MiniRocket =====
    rocket_param_space = {
        **temporal_extractor_grid_space,
        "classifier__base_estimator__num_kernels": [200],
        "classifier__base_estimator__alpha": [1e3],
        "classifier__base_estimator__feature_selection_percentile": [50],
        "classifier__base_estimator__class_weight": ["balanced"],
    }

In [ ]:
# ============================================================
# MODEL TRAINING & EVALUATION LOOP (FIXED FOR XGBOOST/DEEP FOREST)
# ============================================================

from sklearn.preprocessing import LabelEncoder

results_list = []
fitted_models = {}

# -------------------------------------------------------------
# Helper: encode/decode for classifiers that don't support strings
# -------------------------------------------------------------
# (We'll apply this only to XGBoost and Deep Forest)

# -------------------------------------------------------------
# 1. Random Forest (works with strings)
# -------------------------------------------------------------
if TRAIN_RF:
    print("\n--- Training: Random Forest ---")
    rf_pipe = Pipeline([
        ("extractor", build_feature_extractor("tabular_honeycomb")),
        ("classifier", ManyToOneWrapper(
            base_estimator=RandomForestClassifier(random_state=random_state, n_jobs=-1),
            target=target_col
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rf_search = BayesSearchCV(
            rf_pipe, rf_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        rf_search = GridSearchCV(
            rf_pipe, rf_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    rf_search.fit(X_train, y_train, groups=groups)
    fitted_models["Random Forest"] = rf_search.best_estimator_
    
    y_pred_rf = rf_search.predict(X_test)
    from base_utils_qwen import evaluate_holdout
    eval_dict_rf = evaluate_holdout(y_test, y_pred_rf, target_col=target_col, verbose=True)
    score_rf = eval_dict_rf['competition_score']
    
    print(f"Random Forest Best CV Score: {rf_search.best_score_:.4f} | Holdout Score: {score_rf:.4f}")
    print(f"Best Params: {rf_search.best_params_}")
    
    results_list.append({
        "Model": "Random Forest",
        "CV Score": rf_search.best_score_,
        "Holdout Score": score_rf,
        "Best Params": rf_search.best_params_,
    })

# -------------------------------------------------------------
# 2. XGBoost (needs label encoding)
# -------------------------------------------------------------
if TRAIN_XGB:
    print("\n--- Training: XGBoost ---")
    
    # Encode the target labels
    le = LabelEncoder()
    y_train_encoded = y_train.copy()
    y_train_encoded[target_col] = le.fit_transform(y_train[target_col])
    
    # Build pipeline with encoded labels
    xgb_pipe = Pipeline([
        ("extractor", build_feature_extractor("tabular_honeycomb")),
        ("classifier", ManyToOneWrapper(
            base_estimator=xgb.XGBClassifier(
                objective="multi:softprob", 
                eval_metric="mlogloss", 
                random_state=random_state,
                n_jobs=-1,
                use_label_encoder=False
            ),
            target=target_col
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        xgb_search = BayesSearchCV(
            xgb_pipe, xgb_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        xgb_search = GridSearchCV(
            xgb_pipe, xgb_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    # Fit with encoded labels (still passing groups for stratification)
    xgb_search.fit(X_train, y_train_encoded, groups=groups)
    fitted_models["XGBoost"] = xgb_search.best_estimator_
    
    # Predict and decode back to strings
    y_pred_encoded = xgb_search.predict(X_test)
    y_pred_xgb = le.inverse_transform(y_pred_encoded)
    
    eval_dict_xgb = evaluate_holdout(y_test, y_pred_xgb, target_col=target_col, verbose=True)
    score_xgb = eval_dict_xgb['competition_score']
    
    print(f"XGBoost Best CV Score: {xgb_search.best_score_:.4f} | Holdout Score: {score_xgb:.4f}")
    print(f"Best Params: {xgb_search.best_params_}")
    
    results_list.append({
        "Model": "XGBoost",
        "CV Score": xgb_search.best_score_,
        "Holdout Score": score_xgb,
        "Best Params": xgb_search.best_params_,
    })

# -------------------------------------------------------------
# 3. Deep Forest (needs label encoding)
# -------------------------------------------------------------
if TRAIN_DEEP_FOREST and DEEP_FOREST_AVAILABLE:
    print("\n--- Training: Deep Forest ---")
    
    # Encode the target labels (reuse same encoder)
    le_df = LabelEncoder()
    y_train_encoded_df = y_train.copy()
    y_train_encoded_df[target_col] = le_df.fit_transform(y_train[target_col])
    
    df_pipe = Pipeline([
        ("extractor", build_feature_extractor("tabular_honeycomb")),
        ("classifier", ManyToOneWrapper(
            base_estimator=CascadeForestClassifier(random_state=random_state, n_jobs=-1),
            target=target_col
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        df_search = BayesSearchCV(
            df_pipe, df_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        df_search = GridSearchCV(
            df_pipe, df_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    df_search.fit(X_train, y_train_encoded_df, groups=groups)
    fitted_models["Deep Forest"] = df_search.best_estimator_
    
    y_pred_encoded_df = df_search.predict(X_test)
    y_pred_df = le_df.inverse_transform(y_pred_encoded_df)
    
    eval_dict_df = evaluate_holdout(y_test, y_pred_df, target_col=target_col, verbose=True)
    score_df = eval_dict_df['competition_score']
    
    print(f"Deep Forest Best CV Score: {df_search.best_score_:.4f} | Holdout Score: {score_df:.4f}")
    print(f"Best Params: {df_search.best_params_}")
    
    results_list.append({
        "Model": "Deep Forest",
        "CV Score": df_search.best_score_,
        "Holdout Score": score_df,
        "Best Params": df_search.best_params_,
    })
elif TRAIN_DEEP_FOREST and not DEEP_FOREST_AVAILABLE:
    print("\nSkipping Deep Forest: 'deep-forest' package not installed.")

# -------------------------------------------------------------
# 4. MiniRocket (works with strings via RidgeClassifier)
# -------------------------------------------------------------
if TRAIN_ROCKET and ROCKET_AVAILABLE:
    print("\n--- Training: MiniRocket ---")
    rocket_pipe = make_baseline_pipeline(
        feature_mode="temporal_honeycomb",
        classifier_name="ridge_rocket",
        target_col=target_col,
        feature_kwargs={},
        classifier_kwargs={},
        random_state=random_state,
    )

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        rocket_search = BayesSearchCV(
            rocket_pipe, rocket_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        rocket_search = GridSearchCV(
            rocket_pipe, rocket_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=-1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    rocket_search.fit(X_train, y_train, groups=groups)
    fitted_models["MiniRocket"] = rocket_search.best_estimator_
    
    y_pred_rocket = rocket_search.predict(X_test)
    eval_dict_rocket = evaluate_holdout(y_test, y_pred_rocket, target_col=target_col, verbose=True)
    score_rocket = eval_dict_rocket['competition_score']
    
    print(f"MiniRocket Best CV Score: {rocket_search.best_score_:.4f} | Holdout Score: {score_rocket:.4f}")
    print(f"Best Params: {rocket_search.best_params_}")
    
    results_list.append({
        "Model": "MiniRocket",
        "CV Score": rocket_search.best_score_,
        "Holdout Score": score_rocket,
        "Best Params": rocket_search.best_params_,
    })
elif TRAIN_ROCKET and not ROCKET_AVAILABLE:
    print("\nSkipping MiniRocket: 'sktime' package not installed.")

In [ ]:
# ============================================================
# FINAL SUMMARY + BEST MODEL HOLDOUT EVAL
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "=" * 60)
print("BASELINES SUMMARY")
print("=" * 60)
print(results_df.to_string(index=False))

# Full report for best holdout model
if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"]
    print(f"\nDetailed holdout eval for best model: {best_name}")
    best_model = fitted_models[best_name]
    
    y_pred = best_model.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test[target_col], y_pred))
    
    # Optional: Feature Importance (if extractor exposes it or via permutation)
    if hasattr(best_model.named_steps['classifier'], 'feature_importances_'):
        print("\nTop 10 Feature Importances:")
        # Note: V1CargoExtractor outputs a DataFrame, we can map importances back if needed
        # This is a simplified view; full mapping requires get_feature_names_out()